# ESAP RSSD v2 - scalable sensor-directed response surface sampling

This notebook implements a clean, scalable modernization of the four-stage spatial response surface sampling (SRS/RSSD) workflow in Lesch (2005):

1. standardize sensor variables and form standardized, decorrelated PCA scores;
2. construct a central composite response-surface design in coded PCA space;
3. identify observations close to each theoretical target while retaining geographically separated alternatives; and
4. use reproducible coordinate exchange to maximize the exact geometric mean minimum separation distance (geoMSD), with PCA mismatch used only as a constraint and tie-breaker.

The implementation never constructs a full survey-by-survey distance matrix and never enumerates the Cartesian product of candidate pools. It is intended for projected coordinates in linear units. The default configuration runs a synthetic demonstration so that **Runtime > Run all** succeeds in a fresh Google Colab runtime. For real data, follow the numbered cell-by-cell workflow: upload, optional existing locations, preview, coordinate conversion, ID/X/Y assignment, feature/scaling selection, PCA/design controls, and optimization settings. The plain configuration cell remains available for scripted runs.

Reference: Lesch, S. M. (2005). Sensor-directed response surface sampling designs for characterizing spatial variation in soil properties. *Computers and Electronics in Agriculture*, 46, 153-179. https://doi.org/10.1016/j.compag.2004.11.004


In [ ]:
# @title Import required libraries (run; no editing required) { display-mode: "form" }
# Core dependencies available in a standard Google Colab runtime.
from __future__ import annotations

import gc
import json
import math
import os
import platform
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from importlib import metadata as importlib_metadata
import json
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist, squareform
from scipy.stats import chi2, skew
from sklearn.decomposition import IncrementalPCA, PCA
from sklearn.preprocessing import PowerTransformer, StandardScaler

NOTEBOOK_VERSION = "2.2.0"
np.set_printoptions(precision=4, suppress=True)


## Notebook setup (run once; no editing required)

Edit this one cell. Feature transformations are explicit and default to `none`; supported values are `none`, `log`, and `yeo-johnson`. In `auto` memory mode, ordinary full-data PCA remains preferred unless estimated working memory exceeds a conservative fraction of available RAM. The optional PCA-space candidate prefilter is separate, disabled by default, and marked in metadata whenever it is actually used.


In [ ]:
# @title Initialize internal defaults (run; no editing required) { display-mode: "form" }
@dataclass
class RSSDConfig:
    # Input. The defaults make the notebook executable without external files.
    INPUT_FILE: Optional[str] = None
    EXCEL_SHEET_NAME: Any = 0
    USE_SYNTHETIC_DEMO: bool = True
    SYNTHETIC_ROWS: int = 6000

    # Required column roles. Change these for a real file.
    ID_COLUMN: str = "sample_id"
    X_COLUMN: str = "x"
    Y_COLUMN: str = "y"
    FEATURE_COLUMNS: List[str] = field(
        default_factory=lambda: ["sensor_1", "sensor_2", "sensor_3", "sensor_4"]
    )
    FEATURE_TRANSFORMS: Dict[str, str] = field(default_factory=dict)

    # Response-surface design and sample budget.
    N_SAMPLES: int = 20
    N_DESIGN_COMPONENTS: int = 2
    DESIGN_COVERAGE: float = 0.80
    CENTER_REPLICATES: int = 2
    SAMPLE_BUDGET_MODE: str = "balanced_target_replication"  # or "ccd_exact"

    # Statistical screening and candidate discovery.
    OUTLIER_COVERAGE: float = 0.999
    CANDIDATES_PER_TARGET: int = 3
    PC_CANDIDATE_TOLERANCE: float = 0.15
    MAX_PC_CANDIDATE_TOLERANCE: float = 1.50
    CANDIDATE_TOLERANCE_GROWTH: float = 1.5

    # Optimizer.
    N_OPTIMIZER_STARTS: int = 50
    MAX_OPTIMIZER_SWEEPS: int = 100
    RANDOM_SEED: int = 20250717
    GEOMSD_TIE_REL_TOL: float = 1e-6
    PCA_TIE_TOL: float = 1e-12

    # Memory behavior.
    MEMORY_MODE: str = "auto"  # full, auto, or incremental
    MAX_WORKING_MEMORY_FRACTION: float = 0.45
    INCREMENTAL_BATCH_SIZE: int = 50000
    NUMERIC_DTYPE: str = "float32"

    # Optional approximation for extreme candidate-discovery workloads.
    ALLOW_APPROXIMATE_PREFILTER: bool = False
    APPROX_PREFILTER_TRIGGER_ROWS: int = 750000
    APPROX_PREFILTER_MAX_ROWS: int = 250000
    APPROX_PREFILTER_BINS_PER_PC: int = 8

    # Diagnostics and testing.
    PLOT_MAX_POINTS: int = 50000
    SURVEY_COLOR: str = "C0"
    TARGET_COLOR: str = "C3"
    OUTLIER_COLOR: str = "C1"
    CANDIDATE_COLOR: str = "C2"
    SELECTED_COLOR: str = "C3"
    FOOTPRINT_COLOR: str = "0.5"
    DIAGNOSTIC_CHUNK_SIZE: int = 100000
    RUN_LARGE_STRESS_TEST: bool = False
    STRESS_TEST_ROWS: int = 300000


cfg = RSSDConfig()

# Explicit transform examples for a real run:
# cfg.FEATURE_TRANSFORMS = {"ECa_vertical": "log", "NDVI": "yeo-johnson"}
# cfg.USE_SYNTHETIC_DEMO = False
# cfg.INPUT_FILE = "/content/my_survey.csv"  # or None for Colab upload

print("Internal RSSD defaults initialized. Use the guided cells below; code editing is not required.")


### Internal data and PCA functions

Rows with missing or nonfinite required values are retained in the source dataframe but excluded from analysis. Duplicate IDs are a fatal ambiguity. For duplicate coordinates, the first valid record remains candidate-eligible and later records at exactly the same coordinate are retained but excluded from candidate selection; this prevents nominally distinct samples from having zero geographic separation.


In [ ]:
# @title Load internal data and PCA functions (run; no editing required) { display-mode: "form" }
def make_synthetic_survey(n: int, seed: int) -> pd.DataFrame:
    """Create correlated, spatially structured proxy variables for a runnable demonstration."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 2400.0, n)
    y = rng.uniform(0.0, 1600.0, n)
    z1 = 0.9 * np.sin(x / 310.0) + 0.7 * np.cos(y / 240.0) + rng.normal(0, 0.35, n)
    z2 = 0.75 * z1 + 0.5 * np.sin((x + y) / 420.0) + rng.normal(0, 0.3, n)
    z3 = -0.35 * z1 + 0.9 * np.cos(x / 520.0) + rng.normal(0, 0.45, n)
    z4 = 0.45 * z2 + 0.5 * z3 + rng.normal(0, 0.4, n)
    return pd.DataFrame(
        {
            "sample_id": np.arange(1, n + 1, dtype=np.int64),
            "x": x,
            "y": y,
            "sensor_1": z1,
            "sensor_2": z2,
            "sensor_3": z3,
            "sensor_4": z4,
        }
    )


def load_survey(config: RSSDConfig) -> Tuple[pd.DataFrame, str]:
    """Load CSV/XLS/XLSX input or the deterministic synthetic demonstration."""
    if config.USE_SYNTHETIC_DEMO:
        return make_synthetic_survey(config.SYNTHETIC_ROWS, config.RANDOM_SEED), "synthetic_demo"

    input_path = config.INPUT_FILE
    if input_path is None:
        try:
            from google.colab import files  # type: ignore
        except ImportError as exc:
            raise ValueError(
                "Set cfg.INPUT_FILE to a CSV/XLS/XLSX path outside Colab, or enable the synthetic demo."
            ) from exc
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one CSV or Excel survey file.")
        input_path = next(iter(uploaded))

    path = Path(input_path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xls", ".xlsx", ".xlsm"}:
        try:
            df = pd.read_excel(path, sheet_name=config.EXCEL_SHEET_NAME)
        except ImportError as exc:
            raise ImportError("Excel input requires openpyxl (xlsx/xlsm) or xlrd (xls).") from exc
    else:
        raise ValueError(f"Unsupported input type {suffix!r}; use CSV, XLS, XLSX, or XLSM.")
    return df, str(path)


def validate_config(config: RSSDConfig) -> None:
    """Validate settings that determine statistical and computational behavior."""
    if config.MEMORY_MODE not in {"full", "auto", "incremental"}:
        raise ValueError("MEMORY_MODE must be 'full', 'auto', or 'incremental'.")
    if config.SAMPLE_BUDGET_MODE not in {"ccd_exact", "balanced_target_replication"}:
        raise ValueError("SAMPLE_BUDGET_MODE must be 'ccd_exact' or 'balanced_target_replication'.")
    if not (1 <= config.N_DESIGN_COMPONENTS <= 4):
        raise ValueError(
            "This version supports 1-4 design PCs. For >4 PCs, specify a hybrid or small composite design explicitly."
        )
    if config.N_DESIGN_COMPONENTS == 4:
        warnings.warn(
            "A full 4-D central composite design contains many targets; Lesch notes that hybrid or small composite designs may be preferable for 4-6 variables."
        )
    for name, value in {
        "OUTLIER_COVERAGE": config.OUTLIER_COVERAGE,
        "DESIGN_COVERAGE": config.DESIGN_COVERAGE,
    }.items():
        if not 0.0 < value < 1.0:
            raise ValueError(f"{name} must lie strictly between 0 and 1.")
    if config.CANDIDATES_PER_TARGET < 2:
        raise ValueError("CANDIDATES_PER_TARGET must be at least 2.")
    if config.N_OPTIMIZER_STARTS < 1:
        raise ValueError("N_OPTIMIZER_STARTS must be positive.")


def coordinates_look_geographic(x: np.ndarray, y: np.ndarray, x_name: str, y_name: str) -> bool:
    """Conservatively detect decimal-degree longitude/latitude coordinates."""
    xn, yn = x_name.lower(), y_name.lower()
    name_signal = ("lon" in xn or "longitude" in xn) and ("lat" in yn or "latitude" in yn)
    bounded = np.mean((x >= -180) & (x <= 180) & (y >= -90) & (y <= 90)) >= 0.99
    finite = np.isfinite(x) & np.isfinite(y)
    if not finite.any():
        return False
    xf, yf = x[finite], y[finite]
    decimal_signal = np.mean(np.abs(xf - np.round(xf)) > 1e-6) > 0.8 and np.mean(
        np.abs(yf - np.round(yf)) > 1e-6
    ) > 0.8
    typical_lonlat = (
        np.nanmedian(np.abs(xf)) >= 20
        and np.nanmedian(np.abs(yf)) >= 10
        and np.ptp(xf) <= 20
        and np.ptp(yf) <= 20
    )
    return bool(name_signal or (bounded and decimal_signal and typical_lonlat))


def validate_and_index_data(
    df: pd.DataFrame, config: RSSDConfig
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, Dict[str, Any]]:
    """Validate required data and return valid-row and coordinate-eligibility masks."""
    required = [config.ID_COLUMN, config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]
    missing_columns = [c for c in required if c not in df.columns]
    if missing_columns:
        raise KeyError(f"Missing required columns: {missing_columns}")
    if len(set(config.FEATURE_COLUMNS)) != len(config.FEATURE_COLUMNS):
        raise ValueError("FEATURE_COLUMNS contains duplicates.")
    if config.N_DESIGN_COMPONENTS > len(config.FEATURE_COLUMNS):
        raise ValueError("N_DESIGN_COMPONENTS cannot exceed the number of feature columns.")

    duplicate_id_count = int(df[config.ID_COLUMN].duplicated(keep=False).sum())
    if duplicate_id_count:
        raise ValueError(
            f"Found {duplicate_id_count} rows with duplicated IDs. IDs must be unique before RSSD selection."
        )

    numeric = df[[config.X_COLUMN, config.Y_COLUMN, *config.FEATURE_COLUMNS]].apply(
        pd.to_numeric, errors="coerce"
    )
    arr = numeric.to_numpy(dtype=np.float64, copy=False)
    finite_mask = np.isfinite(arr).all(axis=1)
    missing_nonfinite_counts = {
        c: int((~np.isfinite(numeric[c].to_numpy(dtype=np.float64))).sum())
        for c in numeric.columns
    }
    n_invalid = int((~finite_mask).sum())
    if finite_mask.sum() < max(config.N_SAMPLES, 20):
        raise ValueError("Too few complete finite rows remain for the requested design.")

    valid_features = numeric.loc[finite_mask, config.FEATURE_COLUMNS]
    zero_variance = [c for c in config.FEATURE_COLUMNS if float(valid_features[c].var(ddof=0)) <= 0.0]
    if zero_variance:
        raise ValueError(f"Zero-variance feature columns are not usable: {zero_variance}")

    x = numeric.loc[finite_mask, config.X_COLUMN].to_numpy(dtype=np.float64)
    y = numeric.loc[finite_mask, config.Y_COLUMN].to_numpy(dtype=np.float64)
    if coordinates_look_geographic(x, y, config.X_COLUMN, config.Y_COLUMN):
        raise ValueError(
            "Coordinates strongly resemble longitude/latitude in decimal degrees. Reproject to an appropriate linear-unit CRS (for example, the relevant UTM zone) before computing geoMSD."
        )

    duplicated_coordinates_all = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep=False).to_numpy()
    later_coordinate_duplicate = df.duplicated([config.X_COLUMN, config.Y_COLUMN], keep="first").to_numpy()
    coordinate_eligible = finite_mask & ~later_coordinate_duplicate

    out = df.copy()
    out["rssd_complete_finite"] = finite_mask
    out["rssd_duplicate_coordinate"] = duplicated_coordinates_all
    out["rssd_coordinate_candidate_eligible"] = coordinate_eligible
    report = {
        "input_rows": int(len(df)),
        "complete_finite_rows": int(finite_mask.sum()),
        "invalid_rows_excluded": n_invalid,
        "duplicated_id_rows": duplicate_id_count,
        "duplicated_coordinate_rows": int(duplicated_coordinates_all.sum()),
        "later_duplicate_coordinate_rows_excluded_from_candidates": int(later_coordinate_duplicate.sum()),
        "missing_or_nonfinite_by_column": missing_nonfinite_counts,
        "duplicate_coordinate_handling": (
            "All records are retained; among exact coordinate duplicates, only the first valid record is candidate-eligible."
        ),
    }
    return out, finite_mask, coordinate_eligible, report


def transform_features(
    values: np.ndarray, feature_names: Sequence[str], transforms: Mapping[str, str]
) -> Tuple[np.ndarray, Dict[str, Dict[str, Any]]]:
    """Apply explicit per-feature transformations without automatic choices."""
    transformed = np.asarray(values, dtype=np.float64).copy()
    details: Dict[str, Dict[str, Any]] = {}
    allowed = {"none", "log", "yeo-johnson"}
    unknown_keys = sorted(set(transforms) - set(feature_names))
    if unknown_keys:
        raise KeyError(f"FEATURE_TRANSFORMS contains unknown features: {unknown_keys}")
    for j, name in enumerate(feature_names):
        method = transforms.get(name, "none").lower()
        if method not in allowed:
            raise ValueError(f"Unsupported transform {method!r} for {name}; choose {sorted(allowed)}.")
        if method == "none":
            details[name] = {"method": "none"}
        elif method == "log":
            if np.any(transformed[:, j] <= 0):
                raise ValueError(f"Log transformation requires strictly positive values in {name!r}.")
            transformed[:, j] = np.log(transformed[:, j])
            details[name] = {"method": "log", "base": "natural"}
        else:
            pt = PowerTransformer(method="yeo-johnson", standardize=False)
            transformed[:, [j]] = pt.fit_transform(transformed[:, [j]])
            details[name] = {"method": "yeo-johnson", "lambda": float(pt.lambdas_[0])}
    return transformed.astype(np.float32, copy=False), details


def available_memory_bytes() -> int:
    """Return available RAM when psutil is present, otherwise use a conservative Colab fallback."""
    try:
        import psutil

        return int(psutil.virtual_memory().available)
    except ImportError:
        return 8 * 1024**3


def choose_pca_mode(n_rows: int, n_features: int, config: RSSDConfig) -> Tuple[str, Dict[str, float]]:
    """Choose full or incremental PCA from an explicit working-memory estimate."""
    # Approximate transformed, standardized, work, and score arrays plus estimator overhead.
    estimated = int(n_rows * n_features * (4 + 4 + 8 + 4) + n_rows * config.N_DESIGN_COMPONENTS * 4)
    available = available_memory_bytes()
    safe = int(config.MAX_WORKING_MEMORY_FRACTION * available)
    if config.MEMORY_MODE == "auto":
        mode = "full" if estimated <= safe else "incremental"
    else:
        mode = config.MEMORY_MODE
    return mode, {
        "estimated_working_bytes": estimated,
        "available_memory_bytes": available,
        "safe_memory_bytes": safe,
        "estimated_working_mib": estimated / 1024**2,
    }


def fit_standardized_pca(
    transformed: np.ndarray, config: RSSDConfig
) -> Tuple[np.ndarray, StandardScaler, Any, str, Dict[str, float]]:
    """Standardize features, fit PCA, and divide retained scores by sqrt(explained variance)."""
    n, f = transformed.shape
    mode, memory_report = choose_pca_mode(n, f, config)
    scaler = StandardScaler(copy=True)
    batch = max(config.INCREMENTAL_BATCH_SIZE, f + 1)

    def batch_slices() -> Iterable[Tuple[int, int]]:
        """Yield batches while merging a final batch smaller than the feature count."""
        start = 0
        while start < n:
            end = min(start + batch, n)
            if 0 < n - end < f:
                end = n
            yield start, end
            start = end

    if mode == "full":
        standardized = scaler.fit_transform(transformed).astype(np.float32, copy=False)
        estimator: Any = PCA(n_components=f, svd_solver="full", random_state=config.RANDOM_SEED)
        estimator.fit(standardized)
        retained_raw = (standardized - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T
    else:
        for start, end in batch_slices():
            scaler.partial_fit(transformed[start:end])
        estimator = IncrementalPCA(n_components=f, batch_size=batch)
        for start, end in batch_slices():
            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)
            estimator.partial_fit(xb)
        retained_raw = np.empty((n, config.N_DESIGN_COMPONENTS), dtype=np.float32)
        for start, end in batch_slices():
            xb = scaler.transform(transformed[start:end]).astype(np.float32, copy=False)
            retained_raw[start:end] = (
                (xb - estimator.mean_) @ estimator.components_[: config.N_DESIGN_COMPONENTS].T
            )

    scales = np.sqrt(np.asarray(estimator.explained_variance_[: config.N_DESIGN_COMPONENTS]))
    if np.any(scales <= 0):
        raise ValueError("PCA produced a nonpositive explained variance in a retained component.")
    design_scores = (retained_raw / scales).astype(np.float32, copy=False)
    return design_scores, scaler, estimator, mode, memory_report


def pca_validation_table(scores: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray]:
    """Return means, sample standard deviations, and the correlation matrix."""
    names = [f"PC{i + 1}" for i in range(scores.shape[1])]
    table = pd.DataFrame(
        {"mean": np.mean(scores, axis=0), "sample_sd": np.std(scores, axis=0, ddof=1)}, index=names
    )
    corr = np.corrcoef(scores, rowvar=False)
    corr = np.atleast_2d(corr)
    return table, corr


### Internal spatial design and optimization functions

For selected coordinates, the cKDTree query uses `k=2`; the first neighbor is the site itself and the second is its exact nearest selected neighbor. Therefore the computed geoMSD is the exact computational form of Lesch's Eq. 9 criterion, not an approximation.


In [ ]:
# @title Load internal spatial-design functions (run; no editing required) { display-mode: "form" }
def selected_nearest_neighbor_distances(coords: np.ndarray) -> np.ndarray:
    """Calculate each selected site's exact minimum separation distance."""
    coords = np.asarray(coords, dtype=np.float64)
    if coords.ndim != 2 or coords.shape[0] < 2:
        raise ValueError("At least two selected coordinate pairs are required.")
    distances, _ = cKDTree(coords).query(coords, k=2, workers=-1)
    return np.asarray(distances[:, 1], dtype=np.float64)


def exact_geo_msd(coords: np.ndarray) -> float:
    """Compute exp(mean(log(d_j))) for exact nearest-selected-neighbor distances d_j."""
    d = selected_nearest_neighbor_distances(coords)
    if np.any(d <= 0):
        return 0.0
    return float(np.exp(np.mean(np.log(d))))


def make_base_ccd(p: int, radius: float, center_replicates: int) -> pd.DataFrame:
    """Construct cube, axial, and replicated-center targets on common outer radius R."""
    if p > 4:
        raise ValueError("p > 4 requires a separately specified hybrid or small composite design.")
    rows: List[Dict[str, Any]] = []
    cube_level = radius / math.sqrt(p)
    for number in range(2**p):
        signs = np.array([1.0 if (number >> j) & 1 else -1.0 for j in range(p)])
        row = {"base_target_id": f"cube_{number + 1:02d}", "target_type": "cube"}
        row.update({f"target_PC{j + 1}": float(signs[j] * cube_level) for j in range(p)})
        rows.append(row)
    for axis in range(p):
        for sign, label in [(-1.0, "minus"), (1.0, "plus")]:
            values = np.zeros(p)
            values[axis] = sign * radius
            row = {
                "base_target_id": f"axial_PC{axis + 1}_{label}",
                "target_type": "axial",
            }
            row.update({f"target_PC{j + 1}": float(values[j]) for j in range(p)})
            rows.append(row)
    for c in range(center_replicates):
        row = {"base_target_id": f"center_{c + 1:02d}", "target_type": "center"}
        row.update({f"target_PC{j + 1}": 0.0 for j in range(p)})
        rows.append(row)
    return pd.DataFrame(rows)


def allocate_target_instances(
    base_targets: pd.DataFrame, n_samples: int, mode: str, p: int
) -> Tuple[pd.DataFrame, pd.Series]:
    """Create exact-CCD or deterministically balanced replicated target instances."""
    base_n = len(base_targets)
    if mode == "ccd_exact":
        counts = np.ones(base_n, dtype=int)
        if n_samples != base_n:
            print(f"ccd_exact mode uses {base_n} samples; configured N_SAMPLES={n_samples} is ignored.")
    else:
        if n_samples < base_n:
            raise ValueError(
                f"balanced_target_replication requires N_SAMPLES >= {base_n}, the base CCD size."
            )
        counts = np.full(base_n, n_samples // base_n, dtype=int)
        counts[: n_samples % base_n] += 1

    rows: List[Dict[str, Any]] = []
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    for i, base in base_targets.iterrows():
        for replicate in range(1, int(counts[i]) + 1):
            row = {
                "target_instance_id": f"T{len(rows) + 1:03d}",
                "base_target_id": base["base_target_id"],
                "target_type": base["target_type"],
                "target_replicate_number": replicate,
            }
            row.update({c: float(base[c]) for c in target_cols})
            rows.append(row)
    instances = pd.DataFrame(rows)
    replication = instances.groupby("base_target_id", sort=False).size().rename("replication_count")
    return instances, replication


def approximate_pca_prefilter(
    eligible_indices: np.ndarray,
    scores: np.ndarray,
    targets: np.ndarray,
    config: RSSDConfig,
) -> Tuple[np.ndarray, bool, Dict[str, Any]]:
    """Optional reproducible occupancy prefilter that explicitly preserves tails and target neighborhoods."""
    eligible_indices = np.asarray(eligible_indices, dtype=int)
    if (
        not config.ALLOW_APPROXIMATE_PREFILTER
        or len(eligible_indices) <= config.APPROX_PREFILTER_TRIGGER_ROWS
    ):
        return eligible_indices, False, {"reason": "disabled_or_below_trigger"}

    rng = np.random.default_rng(config.RANDOM_SEED)
    z = scores[eligible_indices]
    p = z.shape[1]
    bins = config.APPROX_PREFILTER_BINS_PER_PC
    bin_ids = np.zeros((len(z), p), dtype=np.int16)
    for j in range(p):
        edges = np.quantile(z[:, j], np.linspace(0, 1, bins + 1)[1:-1])
        bin_ids[:, j] = np.digitize(z[:, j], np.unique(edges), right=False)
    multipliers = (bins ** np.arange(p)).astype(np.int64)
    codes = (bin_ids.astype(np.int64) * multipliers).sum(axis=1)
    occupied = np.unique(codes)
    per_bin_cap = max(1, math.ceil(config.APPROX_PREFILTER_MAX_ROWS / len(occupied)))
    kept_local: List[np.ndarray] = []
    for code_value in occupied:
        members = np.flatnonzero(codes == code_value)
        if len(members) > per_bin_cap:
            members = np.sort(rng.choice(members, size=per_bin_cap, replace=False))
        kept_local.append(members)

    # Explicitly preserve univariate tails.
    tail_n = min(1000, len(z))
    tail_local = []
    for j in range(p):
        order = np.argsort(z[:, j], kind="stable")
        tail_local.extend([order[:tail_n], order[-tail_n:]])

    # Explicitly preserve every observation in each initial target neighborhood.
    tree = cKDTree(z)
    target_local: List[int] = []
    for target in np.unique(targets, axis=0):
        target_local.extend(tree.query_ball_point(target, r=config.PC_CANDIDATE_TOLERANCE))

    local = np.unique(
        np.concatenate([*kept_local, *tail_local, np.asarray(target_local, dtype=int)])
    )
    kept = eligible_indices[local]
    report = {
        "input_eligible_rows": int(len(eligible_indices)),
        "prefilter_rows": int(len(kept)),
        "occupied_pca_strata": int(len(occupied)),
        "per_stratum_cap": int(per_bin_cap),
        "target_neighborhood_rows_explicitly_preserved": int(len(np.unique(target_local))),
    }
    return kept, True, report


def candidate_pool_for_target(
    target: np.ndarray,
    pc_tree: cKDTree,
    search_scores: np.ndarray,
    search_coords: np.ndarray,
    search_global_indices: np.ndarray,
    desired: int,
    config: RSSDConfig,
) -> Tuple[np.ndarray, np.ndarray, float, int]:
    """Retain the closest site, then geographically separated sites within an expanding PC tolerance."""
    tolerance = config.PC_CANDIDATE_TOLERANCE
    local = pc_tree.query_ball_point(target, r=tolerance)
    expansions = 0
    while len(local) < desired and tolerance < config.MAX_PC_CANDIDATE_TOLERANCE:
        tolerance = min(
            config.MAX_PC_CANDIDATE_TOLERANCE,
            tolerance * config.CANDIDATE_TOLERANCE_GROWTH,
        )
        local = pc_tree.query_ball_point(target, r=tolerance)
        expansions += 1

    if len(local) < desired:
        k = min(max(desired, 1), len(search_scores))
        nearest_d, nearest_i = pc_tree.query(target, k=k)
        local = np.atleast_1d(nearest_i).astype(int).tolist()
        tolerance = max(tolerance, float(np.max(np.atleast_1d(nearest_d))))
        warnings.warn(
            f"Target required nearest-k fallback; final PCA tolerance expanded to {tolerance:.4f}."
        )
    if len(local) == 0:
        raise RuntimeError("No candidate observations were found for a response-surface target.")

    local_arr = np.asarray(local, dtype=int)
    pc_dist = np.linalg.norm(search_scores[local_arr] - target, axis=1)
    stable = np.lexsort((search_global_indices[local_arr], pc_dist))
    local_arr = local_arr[stable]
    pc_dist = pc_dist[stable]

    selected_positions = [0]
    remaining = np.ones(len(local_arr), dtype=bool)
    remaining[0] = False
    while len(selected_positions) < min(desired, len(local_arr)):
        rem_pos = np.flatnonzero(remaining)
        retained_coords = search_coords[local_arr[np.asarray(selected_positions)]]
        candidate_coords = search_coords[local_arr[rem_pos]]
        min_geo = np.full(len(rem_pos), np.inf)
        for retained_coord in retained_coords:
            min_geo = np.minimum(min_geo, np.linalg.norm(candidate_coords - retained_coord, axis=1))
        # Max geographic separation, then lower PCA distance, then lower original row index.
        order = np.lexsort(
            (
                search_global_indices[local_arr[rem_pos]],
                pc_dist[rem_pos],
                -min_geo,
            )
        )
        chosen = int(rem_pos[order[0]])
        selected_positions.append(chosen)
        remaining[chosen] = False

    positions = np.asarray(selected_positions, dtype=int)
    return (
        search_global_indices[local_arr[positions]],
        pc_dist[positions],
        float(tolerance),
        expansions,
    )


def discover_candidate_pools(
    target_instances: pd.DataFrame,
    search_indices: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
    pool_multiplier: int = 1,
) -> Tuple[List[np.ndarray], List[np.ndarray], pd.DataFrame, List[float]]:
    """Build target-specific candidate pools without survey-wide geographic pairwise distances."""
    p = config.N_DESIGN_COMPONENTS
    target_cols = [f"target_PC{j + 1}" for j in range(p)]
    targets = target_instances[target_cols].to_numpy(dtype=np.float64)
    search_scores = scores[search_indices]
    search_coords = coords[search_indices]
    pc_tree = cKDTree(search_scores)

    # Identical theoretical targets need enough shared alternatives to support unique field sites.
    target_keys = [tuple(row) for row in np.round(targets, 12)]
    multiplicities = {key: target_keys.count(key) for key in set(target_keys)}
    pools: List[np.ndarray] = []
    pool_distances: List[np.ndarray] = []
    tolerances: List[float] = []
    records: List[Dict[str, Any]] = []
    for t, target in enumerate(targets):
        desired = max(
            config.CANDIDATES_PER_TARGET * pool_multiplier,
            multiplicities[target_keys[t]],
        )
        indices, distances, tolerance, expansions = candidate_pool_for_target(
            target,
            pc_tree,
            search_scores,
            search_coords,
            search_indices,
            desired,
            config,
        )
        pools.append(indices)
        pool_distances.append(distances)
        tolerances.append(tolerance)
        for rank, (index, distance) in enumerate(zip(indices, distances), start=1):
            record = {
                "target_position": t,
                "target_instance_id": target_instances.iloc[t]["target_instance_id"],
                "base_target_id": target_instances.iloc[t]["base_target_id"],
                "target_type": target_instances.iloc[t]["target_type"],
                "candidate_analysis_index": int(index),
                "candidate_rank": rank,
                "pca_target_distance": float(distance),
                "final_tolerance_used": tolerance,
                "tolerance_expansions": expansions,
            }
            record.update({c: float(target_instances.iloc[t][c]) for c in target_cols})
            records.append(record)
    return pools, pool_distances, pd.DataFrame(records), tolerances


def assignment_from_costs(
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    rng: Optional[np.random.Generator] = None,
) -> Optional[np.ndarray]:
    """Use a minimum-cost bipartite assignment; random costs produce reproducible alternative starts."""
    union = np.unique(np.concatenate(pools))
    if len(union) < len(pools):
        return None
    column = {int(index): j for j, index in enumerate(union)}
    prohibitive = 1e12
    cost = np.full((len(pools), len(union)), prohibitive, dtype=np.float64)
    for i, (pool, distances) in enumerate(zip(pools, pool_distances)):
        values = distances if rng is None else rng.random(len(pool))
        for index, value in zip(pool, values):
            cost[i, column[int(index)]] = float(value)
    rows, cols = linear_sum_assignment(cost)
    if len(rows) != len(pools) or np.any(cost[rows, cols] >= prohibitive):
        return None
    selected = np.empty(len(pools), dtype=int)
    selected[rows] = union[cols]
    return selected


def matching_metrics(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray) -> Tuple[float, float]:
    """Return mean and maximum Euclidean target mismatch in standardized PC units."""
    distances = np.linalg.norm(scores[selected] - targets, axis=1)
    return float(np.mean(distances)), float(np.max(distances))


def design_rank(selected: np.ndarray, targets: np.ndarray, scores: np.ndarray, coords: np.ndarray) -> Tuple[float, float, float]:
    """Return geoMSD, mean mismatch, and maximum mismatch for lexicographic comparison."""
    mean_pc, max_pc = matching_metrics(selected, targets, scores)
    return exact_geo_msd(coords[selected]), mean_pc, max_pc


def rank_is_better(
    candidate: Tuple[float, float, float],
    incumbent: Tuple[float, float, float],
    config: RSSDConfig,
) -> bool:
    """Compare valid designs: geoMSD first, then mean and maximum PCA mismatch only on ties."""
    geo_tol = config.GEOMSD_TIE_REL_TOL * max(1.0, abs(candidate[0]), abs(incumbent[0]))
    if candidate[0] > incumbent[0] + geo_tol:
        return True
    if abs(candidate[0] - incumbent[0]) <= geo_tol:
        if candidate[1] < incumbent[1] - config.PCA_TIE_TOL:
            return True
        if abs(candidate[1] - incumbent[1]) <= config.PCA_TIE_TOL:
            return candidate[2] < incumbent[2] - config.PCA_TIE_TOL
    return False


def coordinate_exchange(
    start: np.ndarray,
    pools: Sequence[np.ndarray],
    targets: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Optimize one site at a time; no Cartesian product of candidate pools is generated."""
    selected = np.asarray(start, dtype=int).copy()
    if len(np.unique(selected)) != len(selected):
        raise ValueError("Coordinate exchange requires a unique starting assignment.")
    initial_rank = design_rank(selected, targets, scores, coords)
    current_rank = initial_rank
    accepted = 0
    sweeps = 0
    for sweep in range(1, config.MAX_OPTIMIZER_SWEEPS + 1):
        accepted_this_sweep = 0
        for position in rng.permutation(len(selected)):
            used_elsewhere = set(np.delete(selected, position).tolist())
            best_index = int(selected[position])
            best_rank = current_rank
            for candidate_index in pools[position]:
                candidate_index = int(candidate_index)
                if candidate_index == selected[position] or candidate_index in used_elsewhere:
                    continue
                proposal = selected.copy()
                proposal[position] = candidate_index
                proposal_rank = design_rank(proposal, targets, scores, coords)
                if rank_is_better(proposal_rank, best_rank, config):
                    best_index, best_rank = candidate_index, proposal_rank
            if best_index != selected[position]:
                selected[position] = best_index
                current_rank = best_rank
                accepted += 1
                accepted_this_sweep += 1
        sweeps = sweep
        if accepted_this_sweep == 0:
            break
    return selected, {
        "initial_geoMSD": initial_rank[0],
        "final_geoMSD": current_rank[0],
        "accepted_swaps": accepted,
        "sweeps": sweeps,
        "final_mean_pca_target_distance": current_rank[1],
        "final_max_pca_target_distance": current_rank[2],
    }


def optimize_multiple_starts(
    minimum_distance_start: np.ndarray,
    pools: Sequence[np.ndarray],
    pool_distances: Sequence[np.ndarray],
    targets: np.ndarray,
    scores: np.ndarray,
    coords: np.ndarray,
    config: RSSDConfig,
) -> Tuple[np.ndarray, pd.DataFrame]:
    """Run one minimum-mismatch and reproducible random unique starts, returning the best design."""
    rng = np.random.default_rng(config.RANDOM_SEED)
    summaries: List[Dict[str, Any]] = []
    best: Optional[np.ndarray] = None
    best_rank: Optional[Tuple[float, float, float]] = None
    for start_number in range(1, config.N_OPTIMIZER_STARTS + 1):
        start = minimum_distance_start.copy() if start_number == 1 else assignment_from_costs(
            pools, pool_distances, rng
        )
        if start is None:
            raise RuntimeError("Unable to generate a unique random assignment from candidate pools.")
        optimized, summary = coordinate_exchange(start, pools, targets, scores, coords, config, rng)
        summary["start_number"] = start_number
        summary["start_type"] = "minimum_pca_distance" if start_number == 1 else "random_unique_assignment"
        summaries.append(summary)
        rank = design_rank(optimized, targets, scores, coords)
        if best is None or rank_is_better(rank, best_rank, config):  # type: ignore[arg-type]
            best, best_rank = optimized.copy(), rank
    if best is None:
        raise RuntimeError("Optimizer produced no design.")
    return best, pd.DataFrame(summaries)


### Internal diagnostics and acceptance tests


In [ ]:
# @title Load diagnostics and acceptance tests (run; no editing required) { display-mode: "form" }
def second_order_matrix(scores: np.ndarray) -> Tuple[np.ndarray, List[str]]:
    """Build intercept, linear, squared, and pairwise-interaction columns."""
    z = np.asarray(scores, dtype=np.float64)
    n, p = z.shape
    columns = [np.ones(n)]
    names = ["Intercept"]
    for j in range(p):
        columns.append(z[:, j])
        names.append(f"PC{j + 1}")
    for j in range(p):
        columns.append(z[:, j] ** 2)
        names.append(f"PC{j + 1}^2")
    for j in range(p):
        for k in range(j + 1, p):
            columns.append(z[:, j] * z[:, k])
            names.append(f"PC{j + 1}:PC{k + 1}")
    return np.column_stack(columns), names


def regression_design_diagnostics(
    selected_scores: np.ndarray,
    population_scores: np.ndarray,
    chunk_size: int,
) -> Tuple[Dict[str, Any], np.ndarray]:
    """Calculate rank, condition, leverage, and chunked average relative prediction variance."""
    x, names = second_order_matrix(selected_scores)
    rank = int(np.linalg.matrix_rank(x))
    n_columns = x.shape[1]
    condition = float(np.linalg.cond(x))
    result: Dict[str, Any] = {
        "model_terms": names,
        "matrix_rows": int(x.shape[0]),
        "matrix_columns": int(n_columns),
        "matrix_rank": rank,
        "full_rank": rank == n_columns,
        "condition_number": condition,
    }
    if rank != n_columns:
        result.update(
            {
                "maximum_leverage": None,
                "average_relative_prediction_variance": None,
                "warning": "Rank deficient: inverse-based leverage and avePVar were not calculated.",
            }
        )
        return result, np.full(len(selected_scores), np.nan)

    xtx_inv = np.linalg.inv(x.T @ x)
    leverage = np.einsum("ij,jk,ik->i", x, xtx_inv, x)
    total = 0.0
    count = 0
    for start in range(0, len(population_scores), chunk_size):
        xp, _ = second_order_matrix(population_scores[start : start + chunk_size])
        h = np.einsum("ij,jk,ik->i", xp, xtx_inv, xp)
        total += float(np.sum(1.0 + h))
        count += len(h)
    result.update(
        {
            "maximum_leverage": float(np.max(leverage)),
            "average_relative_prediction_variance": total / count,
        }
    )
    return result, leverage


def json_ready(value: Any) -> Any:
    """Recursively convert NumPy/Pandas values to JSON-safe Python values."""
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    return value


def package_versions() -> Dict[str, str]:
    """Record software versions needed to reproduce a run."""
    packages = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "openpyxl"]
    versions = {"python": platform.python_version()}
    for package in packages:
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = "not_installed"
    return versions


def run_unit_validations(seed: int) -> None:
    """Validate exact geoMSD and standardized PCA scores on small synthetic data."""
    coords = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 4.0], [3.0, 4.0], [8.0, 1.0]])
    kd_value = exact_geo_msd(coords)
    dense = squareform(pdist(coords))
    np.fill_diagonal(dense, np.inf)
    brute_value = float(np.exp(np.mean(np.log(np.min(dense, axis=1)))))
    np.testing.assert_allclose(kd_value, brute_value, rtol=1e-13, atol=1e-13)

    rng = np.random.default_rng(seed)
    latent = rng.normal(size=(4000, 3))
    mixed = latent @ np.array([[1.0, 0.8, -0.4], [0.2, 1.4, 0.7], [-0.5, 0.1, 1.1]])
    xs = StandardScaler().fit_transform(mixed)
    estimator = PCA(n_components=3, svd_solver="full").fit(xs)
    whitened = estimator.transform(xs) / np.sqrt(estimator.explained_variance_)
    np.testing.assert_allclose(np.mean(whitened, axis=0), 0.0, atol=1e-12)
    np.testing.assert_allclose(np.std(whitened, axis=0, ddof=1), 1.0, atol=1e-12)
    np.testing.assert_allclose(np.corrcoef(whitened, rowvar=False), np.eye(3), atol=1e-12)
    print(f"Unit validations passed: exact geoMSD={kd_value:.12g}; PCA scores standardized/decorrelated.")


validate_config(cfg)
run_unit_validations(cfg.RANDOM_SEED)


## 1. Guided Google Colab workflow

Run the following cells in order. Each stage presents buttons, selectors, sliders, previews, or text fields. No code editing is required. After the final **Apply optimization settings** button reports that the workflow is ready, continue to **Run the scalable RSSD analysis**.


### 3.1 Upload the survey file

Choose the primary CSV or Excel survey file. A second file can be loaded and previewed for reference, but the first file is the RSSD survey input. The clear buttons cancel/reset a selected upload. The synthetic button is only for demonstration.


In [ ]:
# @title 1. Choose survey file(s), cancel/reset, or load the demonstration
import io

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("ipywidgets is required and is included in Google Colab.") from exc

try:
    from google.colab import output as colab_output  # type: ignore

    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

SURVEY_DF: Optional[pd.DataFrame] = None
SECOND_DF: Optional[pd.DataFrame] = None
SURVEY_INPUT_LABEL: Optional[str] = None
WORKFLOW_CONFIG_APPLIED = False

primary_upload = widgets.FileUpload(
    accept=".csv,.xls,.xlsx,.xlsm", multiple=False, description="Choose first file"
)
second_upload = widgets.FileUpload(
    accept=".csv,.xls,.xlsx,.xlsm", multiple=False, description="Choose second file"
)
load_uploads_button = widgets.Button(description="Load selected file(s)", button_style="success")
clear_primary_button = widgets.Button(description="Cancel/reset first", button_style="warning")
clear_second_button = widgets.Button(description="Cancel/reset second", button_style="warning")
demo_button = widgets.Button(description="Use synthetic demo")
upload_output = widgets.Output()


def _upload_name_content(upload: widgets.FileUpload) -> Tuple[str, bytes]:
    value = upload.value
    if not value:
        raise ValueError("No file selected.")
    if isinstance(value, dict):
        name, item = next(iter(value.items()))
        content = item.get("content", item) if isinstance(item, dict) else item
    else:
        item = value[0]
        name, content = item["name"], item["content"]
    return str(name), bytes(content)


def _read_uploaded_dataframe(name: str, content: bytes) -> pd.DataFrame:
    suffix = Path(name).suffix.lower()
    stream = io.BytesIO(content)
    if suffix == ".csv":
        return pd.read_csv(stream)
    if suffix in {".xls", ".xlsx", ".xlsm"}:
        return pd.read_excel(stream)
    raise ValueError("Use CSV, XLS, XLSX, or XLSM.")


def _clear_upload(upload: widgets.FileUpload) -> None:
    try:
        upload.value = ()
    except Exception:
        upload.value.clear()
        upload._counter = 0


def _load_uploads(_: widgets.Button) -> None:
    global SURVEY_DF, SECOND_DF, SURVEY_INPUT_LABEL, WORKFLOW_CONFIG_APPLIED
    with upload_output:
        upload_output.clear_output(wait=True)
        try:
            name, content = _upload_name_content(primary_upload)
            SURVEY_DF = _read_uploaded_dataframe(name, content)
            SURVEY_INPUT_LABEL = f"uploaded:{name}"
            SECOND_DF = None
            if second_upload.value:
                second_name, second_content = _upload_name_content(second_upload)
                SECOND_DF = _read_uploaded_dataframe(second_name, second_content)
                print(f"Second/reference file: {second_name} ({len(SECOND_DF):,} rows)")
            WORKFLOW_CONFIG_APPLIED = False
            print(f"First RSSD file loaded: {name} ({len(SURVEY_DF):,} rows × {SURVEY_DF.shape[1]} columns)")
        except Exception as exc:
            print(f"Upload error: {type(exc).__name__}: {exc}")


def _load_demo(_: widgets.Button) -> None:
    global SURVEY_DF, SECOND_DF, SURVEY_INPUT_LABEL, WORKFLOW_CONFIG_APPLIED
    with upload_output:
        upload_output.clear_output(wait=True)
        SURVEY_DF = make_synthetic_survey(cfg.SYNTHETIC_ROWS, cfg.RANDOM_SEED)
        SECOND_DF = None
        SURVEY_INPUT_LABEL = "synthetic_demo"
        WORKFLOW_CONFIG_APPLIED = False
        print(f"Synthetic demonstration loaded: {len(SURVEY_DF):,} rows.")


load_uploads_button.on_click(_load_uploads)
clear_primary_button.on_click(lambda _: _clear_upload(primary_upload))
clear_second_button.on_click(lambda _: _clear_upload(second_upload))
demo_button.on_click(_load_demo)

display(
    widgets.VBox(
        [
            widgets.HBox([primary_upload, clear_primary_button]),
            widgets.HBox([second_upload, clear_second_button]),
            widgets.HBox([load_uploads_button, demo_button]),
            upload_output,
        ]
    )
)


### 3.2 Enter existing or required target locations (optional)

Use **+ Add location** for sites already known or previously sampled. Enter X/longitude and Y/latitude in the same coordinate system as the survey input. Later, choose whether these locations are only evaluated/overlaid or forced into the final design. Forced locations are matched to unique eligible survey observations and locked to the statistically closest response-surface targets.


In [ ]:
# @title 2. Add or remove existing target locations
preferred_rows: List[Tuple[widgets.Text, widgets.FloatText, widgets.FloatText, widgets.HBox]] = []
preferred_box = widgets.VBox()
add_preferred_button = widgets.Button(description="+ Add location", button_style="info")
clear_preferred_button = widgets.Button(description="Clear all", button_style="warning")


def _refresh_preferred_box() -> None:
    preferred_box.children = tuple(row[3] for row in preferred_rows)


def _remove_preferred(row: Tuple[Any, Any, Any, Any]) -> None:
    if row in preferred_rows:
        preferred_rows.remove(row)
    _refresh_preferred_box()


def _add_preferred(_: Optional[widgets.Button] = None) -> None:
    identifier = widgets.Text(value=f"existing_{len(preferred_rows) + 1}", description="Location ID:")
    x_value = widgets.FloatText(description="X / longitude:", style={"description_width": "initial"})
    y_value = widgets.FloatText(description="Y / latitude:", style={"description_width": "initial"})
    remove = widgets.Button(description="X", layout=widgets.Layout(width="38px"))
    container = widgets.HBox([identifier, x_value, y_value, remove])
    row = (identifier, x_value, y_value, container)
    remove.on_click(lambda _, current=row: _remove_preferred(current))
    preferred_rows.append(row)
    _refresh_preferred_box()


def collect_preferred_raw() -> pd.DataFrame:
    records = []
    for identifier, x_value, y_value, _ in preferred_rows:
        records.append(
            {
                "preferred_location_id": identifier.value or f"existing_{len(records) + 1}",
                "input_x": float(x_value.value),
                "input_y": float(y_value.value),
            }
        )
    return pd.DataFrame(records, columns=["preferred_location_id", "input_x", "input_y"])


add_preferred_button.on_click(_add_preferred)
clear_preferred_button.on_click(lambda _: (preferred_rows.clear(), _refresh_preferred_box()))
display(preferred_box, widgets.HBox([add_preferred_button, clear_preferred_button]))
print("No existing locations are required. Leave this list empty to skip them.")


### 3.3 Preview the first data file

This preview shows the first rows, data types, missing counts, unique counts, and—in the same cell—the optional second file if one was loaded.


In [ ]:
# @title 3. View the first data file and its columns
preview_rows = widgets.IntSlider(value=8, min=3, max=30, description="Preview rows:")
preview_button = widgets.Button(description="Refresh preview", button_style="info")
preview_output = widgets.Output()


def _preview_files(_: Optional[widgets.Button] = None) -> None:
    with preview_output:
        preview_output.clear_output(wait=True)
        if SURVEY_DF is None:
            print("Upload/load the first survey file in step 1.")
            return
        print("FIRST / RSSD SURVEY FILE")
        display(SURVEY_DF.head(preview_rows.value))
        display(
            pd.DataFrame(
                {
                    "dtype": SURVEY_DF.dtypes.astype(str),
                    "missing": SURVEY_DF.isna().sum(),
                    "unique": SURVEY_DF.nunique(dropna=True),
                }
            )
        )
        if SECOND_DF is not None:
            print("SECOND / REFERENCE FILE")
            display(SECOND_DF.head(preview_rows.value))


preview_button.on_click(_preview_files)
display(widgets.HBox([preview_rows, preview_button]), preview_output)
_preview_files()


### 3.4 Coordinate system and decimal-degree to UTM conversion

Select the source X/longitude and Y/latitude columns. If they are decimal degrees, choose automatic UTM detection or a manual zone and hemisphere, then convert. Original longitude/latitude columns are retained and new `_rssd_easting` and `_rssd_northing` columns are added. No zone is hard-coded.


In [ ]:
# @title 4. Select projected coordinates or convert longitude/latitude to UTM
coordinate_mode = widgets.RadioButtons(
    options=[
        ("Already projected in linear units", "projected"),
        ("Decimal degrees (longitude / latitude)", "decimal_degrees"),
    ],
    value="projected",
    description="Input coordinates:",
    style={"description_width": "initial"},
)
coordinate_x_source = widgets.Dropdown(description="X / longitude:")
coordinate_y_source = widgets.Dropdown(description="Y / latitude:")
utm_detection = widgets.RadioButtons(
    options=[("Automatically detect from data centroid", "auto"), ("Choose manually", "manual")],
    value="auto",
    description="UTM selection:",
    style={"description_width": "initial"},
)
utm_zone = widgets.Dropdown(options=list(range(1, 61)), value=13, description="UTM zone:")
utm_hemisphere = widgets.Dropdown(options=["N", "S"], value="N", description="Hemisphere:")
apply_coordinate_system = widgets.Button(description="Apply / convert coordinates", button_style="success")
coordinate_output = widgets.Output()
COORDINATE_MODE_APPLIED: Optional[str] = None
COORDINATE_EPSG: Optional[int] = None


def _guess_column(columns: Sequence[str], candidates: Sequence[str], fallback: str) -> str:
    lookup = {str(column).lower(): str(column) for column in columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    return fallback


def refresh_coordinate_options() -> None:
    if SURVEY_DF is None:
        coordinate_x_source.options = []
        coordinate_y_source.options = []
        return
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    if not numeric:
        raise ValueError("No numeric coordinate columns were found.")
    coordinate_x_source.options = numeric
    coordinate_y_source.options = numeric
    coordinate_x_source.value = _guess_column(
        numeric, ["longitude", "lon", "long", "x", "easting", "utm_x"], numeric[0]
    )
    coordinate_y_source.value = _guess_column(
        numeric,
        ["latitude", "lat", "y", "northing", "utm_y"],
        numeric[min(1, len(numeric) - 1)],
    )


def _epsg_from_zone(zone: int, hemisphere: str) -> int:
    return (32600 if hemisphere == "N" else 32700) + int(zone)


def _auto_utm(lon: np.ndarray, lat: np.ndarray) -> Tuple[int, str, int]:
    lon_center = float(np.nanmedian(lon))
    lat_center = float(np.nanmedian(lat))
    zone = int(np.floor((lon_center + 180.0) / 6.0) + 1)
    zone = min(60, max(1, zone))
    hemisphere = "N" if lat_center >= 0 else "S"
    return zone, hemisphere, _epsg_from_zone(zone, hemisphere)


def transform_lonlat_arrays(lon: np.ndarray, lat: np.ndarray, epsg: int) -> Tuple[np.ndarray, np.ndarray]:
    try:
        from pyproj import Transformer
    except ImportError:
        import subprocess

        print("pyproj is not present; installing the coordinate-conversion dependency once...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyproj"])
        from pyproj import Transformer
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    easting, northing = transformer.transform(lon, lat)
    return np.asarray(easting), np.asarray(northing)


def _apply_coordinates(_: widgets.Button) -> None:
    global SURVEY_DF, COORDINATE_MODE_APPLIED, COORDINATE_EPSG, WORKFLOW_CONFIG_APPLIED
    with coordinate_output:
        coordinate_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("Load the survey file first.")
            x_col, y_col = coordinate_x_source.value, coordinate_y_source.value
            if not x_col or not y_col or x_col == y_col:
                raise ValueError("Choose two distinct numeric coordinate columns.")
            x = pd.to_numeric(SURVEY_DF[x_col], errors="coerce").to_numpy(dtype=float)
            y = pd.to_numeric(SURVEY_DF[y_col], errors="coerce").to_numpy(dtype=float)
            if coordinate_mode.value == "decimal_degrees":
                if not np.all(np.isfinite(x) & np.isfinite(y)):
                    raise ValueError("Longitude/latitude columns contain missing or nonfinite values.")
                if np.any((x < -180) | (x > 180) | (y < -90) | (y > 90)):
                    raise ValueError("Selected decimal-degree values fall outside longitude/latitude bounds.")
                if utm_detection.value == "auto":
                    zone, hemisphere, epsg = _auto_utm(x, y)
                    zone_values = np.floor((x + 180.0) / 6.0).astype(int) + 1
                    if np.unique(zone_values).size > 1:
                        warnings.warn("Survey crosses more than one UTM zone; inspect projection distortion.")
                else:
                    zone, hemisphere = int(utm_zone.value), str(utm_hemisphere.value)
                    epsg = _epsg_from_zone(zone, hemisphere)
                easting, northing = transform_lonlat_arrays(x, y, epsg)
                SURVEY_DF = SURVEY_DF.copy()
                SURVEY_DF["_rssd_easting"] = easting
                SURVEY_DF["_rssd_northing"] = northing
                cfg.X_COLUMN, cfg.Y_COLUMN = "_rssd_easting", "_rssd_northing"
                COORDINATE_EPSG = epsg
                print(f"Converted WGS84 decimal degrees to UTM zone {zone}{hemisphere} (EPSG:{epsg}).")
                print("Added columns: _rssd_easting and _rssd_northing; original coordinates retained.")
            else:
                cfg.X_COLUMN, cfg.Y_COLUMN = str(x_col), str(y_col)
                COORDINATE_EPSG = None
                print("Using existing projected coordinates. Confirm their units and CRS outside the notebook.")
            COORDINATE_MODE_APPLIED = coordinate_mode.value
            WORKFLOW_CONFIG_APPLIED = False
            display(SURVEY_DF[[cfg.X_COLUMN, cfg.Y_COLUMN]].head())
        except Exception as exc:
            print(f"Coordinate error: {type(exc).__name__}: {exc}")


apply_coordinate_system.on_click(_apply_coordinates)
refresh_coordinate_options()
display(
    widgets.VBox(
        [
            coordinate_mode,
            widgets.HBox([coordinate_x_source, coordinate_y_source]),
            utm_detection,
            widgets.HBox([utm_zone, utm_hemisphere]),
            apply_coordinate_system,
            coordinate_output,
        ]
    )
)


### 3.5 Select the sample ID column

Choose an existing unique identifier or create a sequential ID. Duplicate IDs are reported immediately and remain a fatal ambiguity for analysis.


In [ ]:
# @title 5. Select or create the ID column
id_column_selector = widgets.Dropdown(description="Sample ID:", style={"description_width": "initial"})
apply_id_button = widgets.Button(description="Apply ID column", button_style="success")
id_output = widgets.Output()


def refresh_id_options() -> None:
    options = ["Create sequential ID"]
    if SURVEY_DF is not None:
        options.extend([str(c) for c in SURVEY_DF.columns])
    id_column_selector.options = options


def _apply_id(_: widgets.Button) -> None:
    global SURVEY_DF, WORKFLOW_CONFIG_APPLIED
    with id_output:
        id_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("Load the survey file first.")
            if id_column_selector.value == "Create sequential ID":
                SURVEY_DF = SURVEY_DF.copy()
                SURVEY_DF["_rssd_id"] = np.arange(1, len(SURVEY_DF) + 1, dtype=np.int64)
                cfg.ID_COLUMN = "_rssd_id"
            else:
                cfg.ID_COLUMN = str(id_column_selector.value)
            duplicate_rows = int(SURVEY_DF[cfg.ID_COLUMN].duplicated(keep=False).sum())
            if duplicate_rows:
                print(f"ERROR: {duplicate_rows} rows have duplicate IDs. Choose/create a unique ID before analysis.")
            else:
                print(f"ID column set to {cfg.ID_COLUMN!r}; all {len(SURVEY_DF):,} IDs are unique.")
            WORKFLOW_CONFIG_APPLIED = False
        except Exception as exc:
            print(f"ID error: {type(exc).__name__}: {exc}")


apply_id_button.on_click(_apply_id)
refresh_id_options()
display(id_column_selector, apply_id_button, id_output)


### 3.6 Select X/easting and Y/northing; preview geographic space

Select the coordinates used by geoMSD. The preview is interactive and does not calculate a survey-wide distance matrix. If longitude/latitude were converted, choose `_rssd_easting` and `_rssd_northing`.


In [ ]:
# @title 6. Select X/Y columns and preview geographic space
x_column_selector = widgets.Dropdown(description="X / easting:", style={"description_width": "initial"})
y_column_selector = widgets.Dropdown(description="Y / northing:", style={"description_width": "initial"})
geo_color = widgets.Dropdown(
    options=[("Blue", "C0"), ("Green", "C2"), ("Gray", "0.5"), ("Black", "black")],
    value=cfg.SURVEY_COLOR,
    description="Point color:",
)
geo_point_size = widgets.IntSlider(value=4, min=1, max=30, description="Point size:")
apply_xy_button = widgets.Button(description="Apply X/Y", button_style="success")
geo_output = widgets.Output()


def refresh_xy_options() -> None:
    if SURVEY_DF is None:
        x_column_selector.options = []
        y_column_selector.options = []
        return
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns]
    x_column_selector.options = numeric
    y_column_selector.options = numeric
    if cfg.X_COLUMN in numeric:
        x_column_selector.value = cfg.X_COLUMN
    if cfg.Y_COLUMN in numeric:
        y_column_selector.value = cfg.Y_COLUMN


def _plot_geospace(*_: Any) -> None:
    with geo_output:
        geo_output.clear_output(wait=True)
        if SURVEY_DF is None or not x_column_selector.value or not y_column_selector.value:
            print("Load data and choose coordinate columns.")
            return
        x = pd.to_numeric(SURVEY_DF[x_column_selector.value], errors="coerce").to_numpy(float)
        y = pd.to_numeric(SURVEY_DF[y_column_selector.value], errors="coerce").to_numpy(float)
        finite = np.isfinite(x) & np.isfinite(y)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(x[finite], y[finite], s=geo_point_size.value, alpha=0.35, color=geo_color.value, rasterized=True)
        raw_preferred = collect_preferred_raw()
        if len(raw_preferred):
            preferred_x = raw_preferred.input_x.to_numpy(float)
            preferred_y = raw_preferred.input_y.to_numpy(float)
            if COORDINATE_MODE_APPLIED == "decimal_degrees" and COORDINATE_EPSG is not None:
                preferred_x, preferred_y = transform_lonlat_arrays(
                    preferred_x, preferred_y, COORDINATE_EPSG
                )
            ax.scatter(preferred_x, preferred_y, marker="*", s=100, color="C3", label="Entered locations")
            ax.legend()
        ax.set_xlabel(str(x_column_selector.value))
        ax.set_ylabel(str(y_column_selector.value))
        ax.set_title("Geographic-space preview")
        ax.set_aspect("equal", adjustable="box")
        fig.tight_layout()
        plt.show()
        print(f"Finite coordinate rows shown: {int(finite.sum()):,} / {len(finite):,}")


def _apply_xy(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    if x_column_selector.value == y_column_selector.value:
        with geo_output:
            print("Choose distinct X and Y columns.")
        return
    cfg.X_COLUMN = str(x_column_selector.value)
    cfg.Y_COLUMN = str(y_column_selector.value)
    cfg.SURVEY_COLOR = str(geo_color.value)
    WORKFLOW_CONFIG_APPLIED = False
    _plot_geospace()


apply_xy_button.on_click(_apply_xy)
x_column_selector.observe(_plot_geospace, names="value")
y_column_selector.observe(_plot_geospace, names="value")
geo_color.observe(_plot_geospace, names="value")
geo_point_size.observe(_plot_geospace, names="value")
refresh_xy_options()
display(
    widgets.HBox([x_column_selector, y_column_selector]),
    widgets.HBox([geo_color, geo_point_size, apply_xy_button]),
    geo_output,
)
_plot_geospace()


### 3.7 Select sensor features for PCA

Select NDVI alone, NDRE alone, both, or any other numeric proxy columns. Use Ctrl/Cmd-click for multiple selections. Coordinate and ID columns are excluded automatically.


In [ ]:
# @title 7. Select PCA features
feature_selector = widgets.SelectMultiple(
    description="PCA features:", rows=12, layout=widgets.Layout(width="650px"), style={"description_width": "initial"}
)
select_all_features = widgets.Button(description="Select all numeric")
clear_features = widgets.Button(description="Clear selection")
apply_features = widgets.Button(description="Apply feature selection", button_style="success")
feature_output = widgets.Output()


def refresh_feature_options() -> None:
    if SURVEY_DF is None:
        feature_selector.options = []
        return
    excluded = {cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN}
    numeric = [str(c) for c in SURVEY_DF.select_dtypes(include=[np.number]).columns if str(c) not in excluded]
    feature_selector.options = numeric
    preferred = [c for c in numeric if c.lower() in {"ndvi", "ndre"}]
    feature_selector.value = tuple(preferred if preferred else numeric[: min(2, len(numeric))])


def _apply_features(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with feature_output:
        feature_output.clear_output(wait=True)
        features = [str(c) for c in feature_selector.value]
        if not features:
            print("Select at least one numeric sensor feature.")
            return
        cfg.FEATURE_COLUMNS = features
        cfg.N_DESIGN_COMPONENTS = min(cfg.N_DESIGN_COMPONENTS, len(features), 4)
        WORKFLOW_CONFIG_APPLIED = False
        display(SURVEY_DF[features].describe().T)
        print(f"Selected {len(features)} PCA feature(s): {features}")


select_all_features.on_click(lambda _: setattr(feature_selector, "value", tuple(feature_selector.options)))
clear_features.on_click(lambda _: setattr(feature_selector, "value", ()))
apply_features.on_click(_apply_features)
refresh_feature_options()
display(
    feature_selector,
    widgets.HBox([select_all_features, clear_features, apply_features]),
    feature_output,
)


### 3.8 Choose transformation/scaling and inspect feature distributions

Lesch's method requires centered, standardized sensor variables, so `StandardScaler` is always applied. The selector controls the explicit transformation before standardization. Robust or min-max scaling is not substituted because that would change the published coding step.


In [ ]:
# @title 8. Select transformation/scaling and view distributions/correlation
scaling_scheme = widgets.RadioButtons(
    options=[
        ("StandardScaler only (recommended)", "none"),
        ("Yeo-Johnson, then StandardScaler", "yeo-johnson"),
        ("Natural log, then StandardScaler", "log"),
        ("Choose separately for each feature", "custom"),
    ],
    value="none",
    description="Scaling scheme:",
    style={"description_width": "initial"},
)
custom_transform_box = widgets.VBox()
custom_transform_controls: Dict[str, widgets.Dropdown] = {}
apply_scaling_button = widgets.Button(description="Apply and preview", button_style="success")
scaling_output = widgets.Output()


def _refresh_custom_transforms(*_: Any) -> None:
    global custom_transform_controls
    previous = {name: control.value for name, control in custom_transform_controls.items()}
    custom_transform_controls = {}
    controls = []
    for feature in cfg.FEATURE_COLUMNS:
        control = widgets.Dropdown(
            options=["none", "log", "yeo-johnson"],
            value=previous.get(feature, cfg.FEATURE_TRANSFORMS.get(feature, "none")),
            description=feature,
            style={"description_width": "initial"},
        )
        custom_transform_controls[feature] = control
        controls.append(control)
    custom_transform_box.children = tuple(controls) if scaling_scheme.value == "custom" else ()


def _apply_scaling(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with scaling_output:
        scaling_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None or not cfg.FEATURE_COLUMNS:
                raise ValueError("Load data and apply the feature selection first.")
            if scaling_scheme.value == "custom":
                transforms = {name: control.value for name, control in custom_transform_controls.items()}
            else:
                transforms = {name: scaling_scheme.value for name in cfg.FEATURE_COLUMNS}
            values = SURVEY_DF[cfg.FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
            finite = np.isfinite(values.to_numpy(float)).all(axis=1)
            transformed, details = transform_features(
                values.loc[finite].to_numpy(float), cfg.FEATURE_COLUMNS, transforms
            )
            cfg.FEATURE_TRANSFORMS = transforms
            WORKFLOW_CONFIG_APPLIED = False
            n_features = len(cfg.FEATURE_COLUMNS)
            ncols = min(3, n_features)
            nrows = math.ceil(n_features / ncols)
            fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.4 * nrows), squeeze=False)
            for j, feature in enumerate(cfg.FEATURE_COLUMNS):
                axes.flat[j].hist(transformed[:, j], bins=40, color=cfg.SURVEY_COLOR)
                axes.flat[j].set_title(f"{feature} ({transforms[feature]})")
            for ax in axes.flat[n_features:]:
                ax.set_visible(False)
            fig.suptitle("Transformed feature distributions")
            fig.tight_layout()
            plt.show()

            corr = np.corrcoef(transformed, rowvar=False)
            corr = np.atleast_2d(corr)
            fig, ax = plt.subplots(figsize=(max(5, n_features), max(4, n_features * 0.8)))
            image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
            ax.set_xticks(range(n_features), cfg.FEATURE_COLUMNS, rotation=45, ha="right")
            ax.set_yticks(range(n_features), cfg.FEATURE_COLUMNS)
            for i in range(n_features):
                for j in range(n_features):
                    ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center")
            ax.set_title("Selected-feature correlation matrix")
            fig.colorbar(image, ax=ax, shrink=0.8)
            fig.tight_layout()
            plt.show()
            display(pd.DataFrame(details).T)
        except Exception as exc:
            print(f"Scaling/transform error: {type(exc).__name__}: {exc}")


scaling_scheme.observe(_refresh_custom_transforms, names="value")
apply_scaling_button.on_click(_apply_scaling)
_refresh_custom_transforms()
display(scaling_scheme, custom_transform_box, apply_scaling_button, scaling_output)


### 3.9 Select the number of PCA components

Calculate the PCA preview, inspect individual and cumulative explained variance, then choose one to four standardized design PCs. For NDVI-only or NDRE-only, choose one. For a joint NDVI/NDRE design, choose one or two depending on whether both independent directions are needed.


In [ ]:
# @title 9. Calculate PCA preview and select components to retain
calculate_pca_button = widgets.Button(description="Calculate PCA preview", button_style="info")
pca_selector = widgets.IntSlider(value=1, min=1, max=1, step=1, description="Design PCs:")
pca_color = widgets.Dropdown(
    options=[("Blue", "C0"), ("Green", "C2"), ("Purple", "C4"), ("Gray", "0.5")],
    value=cfg.SURVEY_COLOR,
    description="Point color:",
)
apply_pca_button = widgets.Button(description="Apply PCA count", button_style="success")
pca_output = widgets.Output()
PCA_PREVIEW_SCORES: Optional[np.ndarray] = None
PCA_PREVIEW_ESTIMATOR: Any = None


def _calculate_pca_preview(_: Optional[widgets.Button] = None) -> None:
    global PCA_PREVIEW_SCORES, PCA_PREVIEW_ESTIMATOR, WORKFLOW_CONFIG_APPLIED
    with pca_output:
        pca_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None or not cfg.FEATURE_COLUMNS:
                raise ValueError("Load data and apply features/transforms first.")
            values = SURVEY_DF[cfg.FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
            finite = np.isfinite(values).all(axis=1)
            transformed, _ = transform_features(values[finite], cfg.FEATURE_COLUMNS, cfg.FEATURE_TRANSFORMS)
            standardized = StandardScaler().fit_transform(transformed).astype(np.float32, copy=False)
            estimator = PCA(n_components=standardized.shape[1], svd_solver="full", random_state=cfg.RANDOM_SEED)
            raw_scores = estimator.fit_transform(standardized)
            whitened = raw_scores / np.sqrt(estimator.explained_variance_)
            PCA_PREVIEW_ESTIMATOR = estimator
            PCA_PREVIEW_SCORES = whitened[:, : min(4, whitened.shape[1])].astype(np.float32)
            pca_selector.max = PCA_PREVIEW_SCORES.shape[1]
            pca_selector.value = min(max(1, cfg.N_DESIGN_COMPONENTS), pca_selector.max)
            explained = estimator.explained_variance_ratio_
            cumulative = np.cumsum(explained)
            display(
                pd.DataFrame(
                    {
                        "component": np.arange(1, len(explained) + 1),
                        "explained_variance_ratio": explained,
                        "cumulative_explained_variance": cumulative,
                    }
                )
            )
            fig, ax1 = plt.subplots(figsize=(8, 4.5))
            numbers = np.arange(1, len(explained) + 1)
            ax1.bar(numbers, explained, color=pca_color.value, alpha=0.7)
            ax1.set_xlabel("Principal component")
            ax1.set_ylabel("Explained variance ratio")
            ax2 = ax1.twinx()
            ax2.plot(numbers, cumulative, marker="o", color=cfg.TARGET_COLOR)
            ax2.set_ylabel("Cumulative explained variance")
            ax2.set_ylim(0, 1.05)
            ax1.set_title("PCA explained variance")
            fig.tight_layout()
            plt.show()
            WORKFLOW_CONFIG_APPLIED = False
        except Exception as exc:
            print(f"PCA preview error: {type(exc).__name__}: {exc}")


def _apply_pca(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    with pca_output:
        if PCA_PREVIEW_SCORES is None:
            print("Calculate the PCA preview first.")
            return
        cfg.N_DESIGN_COMPONENTS = int(pca_selector.value)
        cfg.SURVEY_COLOR = str(pca_color.value)
        WORKFLOW_CONFIG_APPLIED = False
        print(f"Retained design PCs set to {cfg.N_DESIGN_COMPONENTS}.")


calculate_pca_button.on_click(_calculate_pca_preview)
apply_pca_button.on_click(_apply_pca)
display(widgets.HBox([calculate_pca_button, pca_selector, pca_color, apply_pca_button]), pca_output)


### 3.10 Select outlier threshold, design scale, and sample count

The legacy IQR slider is replaced by a statistically coherent chi-square coverage threshold in standardized PCA space. The **Design coverage** slider is the scale factor: it changes the empirical radius containing the selected proportion of survey observations. Sample count is editable, subject to the full-CCD minimum displayed by the widget.


In [ ]:
# @title 10. Adjust PCA outlier coverage, design radius, sample count, and preview layers
outlier_coverage_control = widgets.FloatSlider(
    value=cfg.OUTLIER_COVERAGE,
    min=0.95,
    max=0.9999,
    step=0.0001,
    readout_format=".4f",
    description="Outlier coverage:",
    style={"description_width": "initial"},
)
design_coverage_control = widgets.FloatSlider(
    value=cfg.DESIGN_COVERAGE,
    min=0.50,
    max=0.99,
    step=0.01,
    description="Design coverage / scale:",
    style={"description_width": "initial"},
)
center_replicates_control = widgets.IntSlider(value=cfg.CENTER_REPLICATES, min=1, max=10, description="Center replicates:", style={"description_width": "initial"})
budget_mode_control = widgets.Dropdown(
    options=[("Balanced target replication", "balanced_target_replication"), ("Exact base CCD", "ccd_exact")],
    value=cfg.SAMPLE_BUDGET_MODE,
    description="Budget mode:",
)
sample_count_control = widgets.BoundedIntText(value=cfg.N_SAMPLES, min=1, max=1000, description="Sample count:")
design_layers = widgets.SelectMultiple(
    options=["All observations", "PCA outliers", "CCD targets"],
    value=("All observations", "PCA outliers", "CCD targets"),
    description="Plot layers:",
    rows=3,
    style={"description_width": "initial"},
)
target_color_control = widgets.Dropdown(
    options=[("Red", "C3"), ("Orange", "C1"), ("Black", "black"), ("Purple", "C4")],
    value=cfg.TARGET_COLOR,
    description="Target color:",
)
outlier_color_control = widgets.Dropdown(
    options=[("Orange", "C1"), ("Red", "C3"), ("Purple", "C4"), ("Black", "black")],
    value=cfg.OUTLIER_COLOR,
    description="Outlier color:",
)
design_point_size = widgets.IntSlider(value=5, min=1, max=30, description="Point size:")
apply_design_button = widgets.Button(description="Apply design settings", button_style="success")
design_preview_output = widgets.Output()
design_minimum_label = widgets.HTML()


def _update_design_minimum(*_: Any) -> None:
    p_value = int(pca_selector.value)
    base_count = 2**p_value + 2 * p_value + int(center_replicates_control.value)
    sample_count_control.min = base_count
    if budget_mode_control.value == "ccd_exact":
        sample_count_control.value = base_count
        sample_count_control.disabled = True
    else:
        sample_count_control.disabled = False
        if sample_count_control.value < base_count:
            sample_count_control.value = base_count
    design_minimum_label.value = (
        f"<b>Full {p_value}-PC CCD minimum: {base_count} samples.</b> "
        f"Current requested count: {sample_count_control.value}."
    )


def _design_preview(*_: Any) -> None:
    with design_preview_output:
        design_preview_output.clear_output(wait=True)
        if PCA_PREVIEW_SCORES is None:
            print("Calculate the PCA preview in step 9 first.")
            return
        p_value = int(pca_selector.value)
        scores = PCA_PREVIEW_SCORES[:, :p_value]
        d2 = np.einsum("ij,ij->i", scores, scores)
        threshold_value = chi2.ppf(outlier_coverage_control.value, df=p_value)
        outliers = d2 > threshold_value
        radius_values = np.linalg.norm(scores, axis=1)
        radius = float(np.quantile(radius_values, design_coverage_control.value))
        base = make_base_ccd(p_value, radius, int(center_replicates_control.value))
        targets, replication = allocate_target_instances(
            base, int(sample_count_control.value), str(budget_mode_control.value), p_value
        )
        target_columns = [f"target_PC{i + 1}" for i in range(p_value)]
        target_values = targets[target_columns].to_numpy(float)
        rng = np.random.default_rng(cfg.RANDOM_SEED)
        plot_n = min(cfg.PLOT_MAX_POINTS, len(scores))
        plot_indices = np.sort(rng.choice(len(scores), plot_n, replace=False))
        fig, ax = plt.subplots(figsize=(8, 5.5 if p_value > 1 else 4.5))
        if p_value == 1:
            if "All observations" in design_layers.value:
                ax.scatter(scores[plot_indices, 0], np.zeros(plot_n), s=design_point_size.value, alpha=0.18, color=pca_color.value, label="Observations")
            if "PCA outliers" in design_layers.value and outliers.any():
                idx = np.flatnonzero(outliers)
                ax.scatter(scores[idx, 0], np.zeros(len(idx)), s=design_point_size.value + 5, color=outlier_color_control.value, label="Outliers")
            if "CCD targets" in design_layers.value:
                ax.scatter(target_values[:, 0], np.zeros(len(target_values)), marker="x", s=75, color=target_color_control.value, label="CCD targets")
            ax.set_yticks([])
            ax.set_xlabel("Standardized PC1")
        else:
            if "All observations" in design_layers.value:
                ax.scatter(scores[plot_indices, 0], scores[plot_indices, 1], s=design_point_size.value, alpha=0.18, color=pca_color.value, label="Observations")
            if "PCA outliers" in design_layers.value and outliers.any():
                idx = np.flatnonzero(outliers)
                ax.scatter(scores[idx, 0], scores[idx, 1], s=design_point_size.value + 5, color=outlier_color_control.value, label="Outliers")
            if "CCD targets" in design_layers.value:
                ax.scatter(target_values[:, 0], target_values[:, 1], marker="x", s=75, color=target_color_control.value, label="CCD targets")
            ax.set_xlabel("Standardized PC1")
            ax.set_ylabel("Standardized PC2")
            ax.set_aspect("equal", adjustable="box")
        ax.set_title("PCA response-surface design preview" + (" (first two PCs)" if p_value > 2 else ""))
        ax.legend()
        fig.tight_layout()
        plt.show()
        display(
            pd.DataFrame(
                {
                    "design_radius_R": [radius],
                    "realized_coverage": [float(np.mean(radius_values <= radius))],
                    "masked_outliers": [int(outliers.sum())],
                    "base_targets": [len(base)],
                    "target_instances": [len(targets)],
                }
            )
        )
        display(replication.to_frame())


def _apply_design(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED
    cfg.N_DESIGN_COMPONENTS = int(pca_selector.value)
    cfg.OUTLIER_COVERAGE = float(outlier_coverage_control.value)
    cfg.DESIGN_COVERAGE = float(design_coverage_control.value)
    cfg.CENTER_REPLICATES = int(center_replicates_control.value)
    cfg.SAMPLE_BUDGET_MODE = str(budget_mode_control.value)
    cfg.N_SAMPLES = int(sample_count_control.value)
    cfg.TARGET_COLOR = str(target_color_control.value)
    cfg.OUTLIER_COLOR = str(outlier_color_control.value)
    WORKFLOW_CONFIG_APPLIED = False
    with design_preview_output:
        print("Design settings applied. Continue to optimization settings.")


for control in [outlier_coverage_control, design_coverage_control, center_replicates_control, budget_mode_control, sample_count_control, design_layers, target_color_control, outlier_color_control, design_point_size]:
    control.observe(_design_preview, names="value")
center_replicates_control.observe(_update_design_minimum, names="value")
budget_mode_control.observe(_update_design_minimum, names="value")
pca_selector.observe(_update_design_minimum, names="value")
apply_design_button.on_click(_apply_design)
_update_design_minimum()
display(
    widgets.VBox(
        [
            widgets.HBox([outlier_coverage_control, design_coverage_control]),
            widgets.HBox([center_replicates_control, budget_mode_control, sample_count_control]),
            design_minimum_label,
            widgets.HBox([design_layers, target_color_control, outlier_color_control, design_point_size]),
            apply_design_button,
            design_preview_output,
        ]
    )
)
_design_preview()


### 3.11 Candidate, optimization, preferred-location, and output controls

These controls replace the legacy weighted-score slider. Candidate tolerance constrains response-surface fidelity; exact geoMSD remains the primary objective. Mean and maximum PCA mismatch are tie-breakers. Existing target locations can be overlaid/evaluated or forced into the design.


In [ ]:
# @title 11. Apply candidate and optimization settings; finalize workflow
candidates_control = widgets.IntSlider(value=cfg.CANDIDATES_PER_TARGET, min=2, max=25, description="Candidates/target:", style={"description_width": "initial"})
tolerance_control = widgets.FloatSlider(value=cfg.PC_CANDIDATE_TOLERANCE, min=0.02, max=1.0, step=0.01, description="Initial PC tolerance:", style={"description_width": "initial"})
max_tolerance_control = widgets.FloatSlider(value=cfg.MAX_PC_CANDIDATE_TOLERANCE, min=0.15, max=3.0, step=0.05, description="Maximum PC tolerance:", style={"description_width": "initial"})
starts_control = widgets.IntSlider(value=cfg.N_OPTIMIZER_STARTS, min=1, max=100, description="Optimizer starts:", style={"description_width": "initial"})
memory_control = widgets.Dropdown(options=["auto", "full", "incremental"], value=cfg.MEMORY_MODE, description="Memory mode:")
approximate_control = widgets.Checkbox(value=cfg.ALLOW_APPROXIMATE_PREFILTER, description="Allow approximate extreme-data prefilter", indent=False)
preferred_mode_control = widgets.Dropdown(
    options=[
        ("Ignore entered locations", "none"),
        ("Evaluate and overlay only", "evaluate_only"),
        ("Force nearest eligible survey sites into final design", "force"),
    ],
    value="evaluate_only" if preferred_rows else "none",
    description="Existing locations:",
    style={"description_width": "initial"},
)
candidate_color_control = widgets.Dropdown(options=[("Green", "C2"), ("Blue", "C0"), ("Purple", "C4"), ("Black", "black")], value=cfg.CANDIDATE_COLOR, description="Candidate color:")
selected_color_control = widgets.Dropdown(options=[("Red", "C3"), ("Orange", "C1"), ("Purple", "C4"), ("Black", "black")], value=cfg.SELECTED_COLOR, description="Selected color:")
apply_optimizer_button = widgets.Button(description="Apply optimization settings", button_style="success", icon="check")
optimizer_output = widgets.Output()
PREFERRED_LOCATIONS = pd.DataFrame(columns=["preferred_location_id", "x", "y"])
PREFERRED_MODE = "none"


def _prepare_preferred_locations() -> pd.DataFrame:
    raw = collect_preferred_raw()
    if len(raw) == 0:
        return pd.DataFrame(columns=["preferred_location_id", "x", "y"])
    if COORDINATE_MODE_APPLIED == "decimal_degrees":
        if COORDINATE_EPSG is None:
            raise ValueError("Apply decimal-degree to UTM conversion before finalizing locations.")
        x, y = transform_lonlat_arrays(
            raw.input_x.to_numpy(float), raw.input_y.to_numpy(float), COORDINATE_EPSG
        )
    else:
        x, y = raw.input_x.to_numpy(float), raw.input_y.to_numpy(float)
    return pd.DataFrame(
        {"preferred_location_id": raw.preferred_location_id.astype(str), "x": x, "y": y}
    )


def _finalize_workflow(_: widgets.Button) -> None:
    global WORKFLOW_CONFIG_APPLIED, PREFERRED_LOCATIONS, PREFERRED_MODE
    with optimizer_output:
        optimizer_output.clear_output(wait=True)
        try:
            if SURVEY_DF is None:
                raise ValueError("No survey dataframe is loaded.")
            required = [cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, *cfg.FEATURE_COLUMNS]
            missing = [column for column in required if column not in SURVEY_DF.columns]
            if missing:
                raise KeyError(f"Required columns not applied or missing: {missing}")
            if COORDINATE_MODE_APPLIED == "decimal_degrees" and cfg.X_COLUMN not in {"_rssd_easting"}:
                raise ValueError("Select the generated UTM easting/northing columns in step 6.")
            cfg.CANDIDATES_PER_TARGET = int(candidates_control.value)
            cfg.PC_CANDIDATE_TOLERANCE = float(tolerance_control.value)
            cfg.MAX_PC_CANDIDATE_TOLERANCE = max(float(max_tolerance_control.value), cfg.PC_CANDIDATE_TOLERANCE)
            cfg.N_OPTIMIZER_STARTS = int(starts_control.value)
            cfg.MEMORY_MODE = str(memory_control.value)
            cfg.ALLOW_APPROXIMATE_PREFILTER = bool(approximate_control.value)
            cfg.CANDIDATE_COLOR = str(candidate_color_control.value)
            cfg.SELECTED_COLOR = str(selected_color_control.value)
            cfg.USE_SYNTHETIC_DEMO = False
            PREFERRED_LOCATIONS = _prepare_preferred_locations()
            PREFERRED_MODE = str(preferred_mode_control.value) if len(PREFERRED_LOCATIONS) else "none"
            if len(PREFERRED_LOCATIONS) > cfg.N_SAMPLES:
                raise ValueError("Existing/forced locations cannot exceed the requested sample count.")
            validate_config(cfg)
            WORKFLOW_CONFIG_APPLIED = True
            display(
                pd.DataFrame(
                    {
                        "setting": ["input", "ID", "X", "Y", "features", "transforms", "design PCs", "samples", "outlier coverage", "design coverage", "candidates/target", "PC tolerance", "optimizer starts", "existing-location mode"],
                        "value": [SURVEY_INPUT_LABEL, cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, cfg.FEATURE_COLUMNS, cfg.FEATURE_TRANSFORMS, cfg.N_DESIGN_COMPONENTS, cfg.N_SAMPLES, cfg.OUTLIER_COVERAGE, cfg.DESIGN_COVERAGE, cfg.CANDIDATES_PER_TARGET, cfg.PC_CANDIDATE_TOLERANCE, cfg.N_OPTIMIZER_STARTS, PREFERRED_MODE],
                    }
                )
            )
            if len(PREFERRED_LOCATIONS):
                display(PREFERRED_LOCATIONS)
            print("WORKFLOW READY. Continue to 'Run the scalable RSSD analysis' below.")
        except Exception as exc:
            WORKFLOW_CONFIG_APPLIED = False
            print(f"Finalization error: {type(exc).__name__}: {exc}")


apply_optimizer_button.on_click(_finalize_workflow)
display(
    widgets.VBox(
        [
            widgets.HBox([candidates_control, tolerance_control, max_tolerance_control]),
            widgets.HBox([starts_control, memory_control, approximate_control]),
            preferred_mode_control,
            widgets.HBox([candidate_color_control, selected_color_control]),
            widgets.HTML("<i>Primary objective: maximize exact geoMSD. PCA mismatch remains constrained and is used only for effective ties.</i>"),
            apply_optimizer_button,
            optimizer_output,
        ]
    )
)


## 2. Run the scalable RSSD analysis

After the previous cell reports **WORKFLOW READY**, run the remaining cells in order. They use the selected dataframe and controls, calculate the corrected scalable design, display diagnostics, compare selected versus full distributions, map the sites, and export the results.


### 4.1 Load and validate the survey; inspect feature distributions

Skewness is reported before and after the explicitly configured transformations. The plots use a reproducible display subset only; all complete observations remain in the numerical analysis unless the optional approximate candidate prefilter is actually activated later.


In [ ]:
run_started = time.perf_counter()
validate_config(cfg)
if globals().get("WORKFLOW_CONFIG_APPLIED", False):
    source_df_raw = SURVEY_DF.copy()
    input_label = str(SURVEY_INPUT_LABEL)
    print("Using the dataframe and settings finalized in the sequential guided workflow.")
else:
    source_df_raw, input_label = load_survey(cfg)
    print("Workflow was not finalized; using the configuration-cell/default synthetic input.")
source_df, valid_mask, coordinate_eligible_mask, validation_report = validate_and_index_data(
    source_df_raw, cfg
)
valid_source_positions = np.flatnonzero(valid_mask)
analysis_to_source = valid_source_positions.copy()
source_to_analysis = np.full(len(source_df), -1, dtype=int)
source_to_analysis[analysis_to_source] = np.arange(len(analysis_to_source))
analysis_coordinate_eligible = coordinate_eligible_mask[analysis_to_source]

original_features = source_df.loc[valid_mask, cfg.FEATURE_COLUMNS].to_numpy(dtype=np.float64)
transformed_features, transformation_details = transform_features(
    original_features, cfg.FEATURE_COLUMNS, cfg.FEATURE_TRANSFORMS
)
skewness_summary = pd.DataFrame(
    {
        "configured_transform": [cfg.FEATURE_TRANSFORMS.get(c, "none") for c in cfg.FEATURE_COLUMNS],
        "raw_skewness": skew(original_features, axis=0, bias=False),
        "transformed_skewness": skew(transformed_features, axis=0, bias=False),
    },
    index=cfg.FEATURE_COLUMNS,
)
print(f"Input: {input_label}; rows={len(source_df):,}; complete analyzed rows={len(analysis_to_source):,}")
display(pd.DataFrame([validation_report]))
display(skewness_summary)

plot_rng = np.random.default_rng(cfg.RANDOM_SEED)
plot_n = min(cfg.PLOT_MAX_POINTS, len(transformed_features))
plot_idx = np.sort(plot_rng.choice(len(transformed_features), size=plot_n, replace=False))
n_features = len(cfg.FEATURE_COLUMNS)
ncols = min(3, n_features)
nrows = math.ceil(n_features / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.5 * nrows), squeeze=False)
for j, name in enumerate(cfg.FEATURE_COLUMNS):
    ax = axes.flat[j]
    ax.hist(transformed_features[plot_idx, j], bins=40, color=cfg.SURVEY_COLOR)
    ax.set_title(f"{name} ({cfg.FEATURE_TRANSFORMS.get(name, 'none')})")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
for ax in axes.flat[n_features:]:
    ax.set_visible(False)
fig.suptitle("Raw or explicitly transformed feature distributions")
fig.tight_layout()
plt.show()


### 4.2 Standardize features, fit PCA, and validate coded design scores


In [ ]:
design_scores, feature_scaler, pca_estimator, pca_mode, memory_report = fit_standardized_pca(
    transformed_features, cfg
)
pca_validation, pca_correlation = pca_validation_table(design_scores)
print(f"PCA mode: {pca_mode}; approximate major-array memory={memory_report['estimated_working_mib']:.1f} MiB")
display(pca_validation)
display(pd.DataFrame(pca_correlation, index=pca_validation.index, columns=pca_validation.index))

mean_abs = float(np.max(np.abs(pca_validation["mean"])))
sd_error = float(np.max(np.abs(pca_validation["sample_sd"] - 1.0)))
off_diag = pca_correlation - np.eye(cfg.N_DESIGN_COMPONENTS)
max_offdiag = float(np.max(np.abs(off_diag)))
if mean_abs > 0.02 or sd_error > 0.02 or max_offdiag > 0.02:
    warnings.warn("Retained PCA scores deviate from the expected standardized/decorrelated validation tolerance.")

explained = np.asarray(pca_estimator.explained_variance_ratio_)
cumulative = np.cumsum(explained)
fig, ax1 = plt.subplots(figsize=(8, 4.8))
components = np.arange(1, len(explained) + 1)
ax1.bar(components, explained, alpha=0.7, label="Explained variance")
ax1.set_xlabel("Principal component")
ax1.set_ylabel("Explained variance ratio")
ax2 = ax1.twinx()
ax2.plot(components, cumulative, marker="o", label="Cumulative variance")
ax2.set_ylabel("Cumulative explained variance")
ax2.set_ylim(0, 1.05)
ax1.set_title("PCA explained and cumulative variance")
fig.tight_layout()
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.bar(pca_validation.index, pca_validation["sample_sd"])
ax1.axhline(1.0, linestyle="--", linewidth=1)
ax1.set_title("Retained PC sample standard deviations")
ax1.set_ylabel("Sample standard deviation")
image = ax2.imshow(pca_correlation, vmin=-1, vmax=1, cmap="coolwarm")
ax2.set_xticks(range(len(pca_validation)), pca_validation.index)
ax2.set_yticks(range(len(pca_validation)), pca_validation.index)
ax2.set_title("Retained PC correlation matrix")
fig.colorbar(image, ax=ax2, shrink=0.8)
fig.suptitle("PCA score standardization diagnostic")
fig.tight_layout()
plt.show()


### 4.3 PCA-space outlier mask and empirical-radius central composite design

Because retained design scores are standardized and decorrelated, squared distance from the PCA origin is screened against a chi-square quantile with degrees of freedom equal to `N_DESIGN_COMPONENTS`. This assumes an approximately elliptical multivariate distribution in the retained standardized PC space. Outliers are flagged and masked from candidate selection; they are not deleted from the source data.


In [ ]:
p = cfg.N_DESIGN_COMPONENTS
coords = source_df.loc[valid_mask, [cfg.X_COLUMN, cfg.Y_COLUMN]].to_numpy(dtype=np.float32)
pc_radius = np.linalg.norm(design_scores, axis=1)
design_radius = float(np.quantile(pc_radius, cfg.DESIGN_COVERAGE))
within_design_radius = pc_radius <= design_radius
actual_design_coverage = float(np.mean(within_design_radius))

pca_d2 = np.einsum("ij,ij->i", design_scores, design_scores)
outlier_threshold_d2 = float(chi2.ppf(cfg.OUTLIER_COVERAGE, df=p))
pca_outlier = pca_d2 > outlier_threshold_d2
candidate_eligible = analysis_coordinate_eligible & ~pca_outlier
if int(candidate_eligible.sum()) < cfg.N_SAMPLES:
    raise ValueError("Outlier and duplicate-coordinate masks leave too few candidate-eligible observations.")

source_df["pca_space_outlier_flag"] = False
source_df.loc[analysis_to_source, "pca_space_outlier_flag"] = pca_outlier

base_targets = make_base_ccd(p, design_radius, cfg.CENTER_REPLICATES)
target_instances, target_replication_counts = allocate_target_instances(
    base_targets, cfg.N_SAMPLES, cfg.SAMPLE_BUDGET_MODE, p
)
target_cols = [f"target_PC{j + 1}" for j in range(p)]
target_array = target_instances[target_cols].to_numpy(dtype=np.float64)

design_report = {
    "design_radius_R": design_radius,
    "actual_proportion_at_or_within_R": actual_design_coverage,
    "cube_points": int((base_targets.target_type == "cube").sum()),
    "axial_points": int((base_targets.target_type == "axial").sum()),
    "center_points": int((base_targets.target_type == "center").sum()),
    "base_design_targets": int(len(base_targets)),
    "target_instances": int(len(target_instances)),
}
display(pd.DataFrame([design_report]))
display(target_replication_counts.to_frame())
print(
    f"PCA-space outliers masked: {int(pca_outlier.sum()):,} / {len(pca_outlier):,}; "
    f"chi-square D2 threshold={outlier_threshold_d2:.4f}."
)

projection_note = " (2-D projection of a higher-dimensional design)" if p > 2 else ""
if p == 1:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.scatter(
        design_scores[plot_idx, 0],
        np.zeros(len(plot_idx)),
        s=6,
        alpha=0.16,
        color=cfg.SURVEY_COLOR,
        label="Valid observations",
    )
    ax.scatter(
        target_array[:, 0],
        np.zeros(len(target_array)),
        marker="x",
        s=75,
        color=cfg.TARGET_COLOR,
        label="Target instances",
    )
    ax.axvline(-design_radius, linestyle="--", linewidth=1)
    ax.axvline(design_radius, linestyle="--", linewidth=1, label="Radius ±R")
    ax.set_xlabel("Standardized PC1")
    ax.set_yticks([])
    ax.set_title("One-dimensional response-surface targets")
    ax.legend()
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4.8))
    normal_idx = plot_idx[~pca_outlier[plot_idx]]
    out_idx = np.flatnonzero(pca_outlier)
    if len(out_idx) > cfg.PLOT_MAX_POINTS:
        out_idx = np.sort(plot_rng.choice(out_idx, cfg.PLOT_MAX_POINTS, replace=False))
    ax.scatter(
        design_scores[normal_idx, 0],
        np.zeros(len(normal_idx)),
        s=6,
        alpha=0.16,
        color=cfg.SURVEY_COLOR,
        label="Not masked",
    )
    if len(out_idx):
        ax.scatter(
            design_scores[out_idx, 0],
            np.zeros(len(out_idx)),
            s=16,
            alpha=0.75,
            color=cfg.OUTLIER_COLOR,
            label="Chi-square outlier",
        )
    ax.set_xlabel("Standardized PC1")
    ax.set_yticks([])
    ax.set_title("One-dimensional PCA-space outlier screening")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(7.5, 6.2))
    ax.scatter(
        design_scores[plot_idx, 0],
        design_scores[plot_idx, 1],
        s=4,
        alpha=0.15,
        color=cfg.SURVEY_COLOR,
        label="Valid observations",
    )
    ax.scatter(
        target_array[:, 0],
        target_array[:, 1],
        marker="x",
        s=70,
        color=cfg.TARGET_COLOR,
        label="Target instances",
    )
    circle = plt.Circle(
        (0, 0), design_radius, fill=False, linestyle="--", linewidth=1, label="Radius R"
    )
    ax.add_patch(circle)
    ax.set_xlabel("Standardized PC1")
    ax.set_ylabel("Standardized PC2")
    ax.set_title("Response-surface targets in PC space" + projection_note)
    ax.set_aspect("equal", adjustable="box")
    ax.legend()
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7.5, 6.2))
    normal_idx = plot_idx[~pca_outlier[plot_idx]]
    out_idx = np.flatnonzero(pca_outlier)
    if len(out_idx) > cfg.PLOT_MAX_POINTS:
        out_idx = np.sort(plot_rng.choice(out_idx, cfg.PLOT_MAX_POINTS, replace=False))
    ax.scatter(
        design_scores[normal_idx, 0],
        design_scores[normal_idx, 1],
        s=4,
        alpha=0.15,
        color=cfg.SURVEY_COLOR,
        label="Not masked",
    )
    if len(out_idx):
        ax.scatter(
            design_scores[out_idx, 0],
            design_scores[out_idx, 1],
            s=12,
            alpha=0.7,
            color=cfg.OUTLIER_COLOR,
            label="Chi-square outlier",
        )
    ax.set_xlabel("Standardized PC1")
    ax.set_ylabel("Standardized PC2")
    ax.set_title("PCA-space outlier screening" + projection_note)
    ax.legend()
    fig.tight_layout()
    plt.show()


### 4.4 Candidate discovery, unique assignment, and multiple-start coordinate exchange

The closest observation is retained first for each target. Subsequent candidates maximize geographic separation from already retained candidates for that target, subject to the target's deterministically expanded PCA tolerance. Candidate-pool overlap is allowed; the bipartite assignment and coordinate-exchange optimizer enforce unique survey observations.


In [ ]:
eligible_indices = np.flatnonzero(candidate_eligible)
search_indices, approximate_prefilter_used, prefilter_report = approximate_pca_prefilter(
    eligible_indices, design_scores, target_array, cfg
)
print(
    f"Candidate discovery rows: {len(search_indices):,}; "
    f"approximate_candidate_prefilter={approximate_prefilter_used}"
)
if approximate_prefilter_used:
    warnings.warn(
        "Approximate PCA-space candidate prefiltering was used; candidate discovery is not mathematically identical to a complete-data search."
    )

minimum_assignment = None
assignment_attempts = 0
for attempt in range(7):
    assignment_attempts = attempt + 1
    candidate_pools, candidate_pool_distances, candidate_associations, target_tolerances = discover_candidate_pools(
        target_instances,
        search_indices,
        design_scores,
        coords,
        cfg,
        pool_multiplier=2**attempt,
    )
    minimum_assignment = assignment_from_costs(candidate_pools, candidate_pool_distances)
    if minimum_assignment is not None:
        break
if minimum_assignment is None:
    raise RuntimeError(
        "No unique target-to-site assignment was possible after deterministic candidate-pool expansion. Increase tolerances, increase candidate counts, or inspect duplicated coordinates."
    )
if assignment_attempts > 1:
    warnings.warn(f"Candidate pools were enlarged through {assignment_attempts} assignment attempts.")

# Match optional entered locations to unique eligible survey observations.
preferred_locations = globals().get(
    "PREFERRED_LOCATIONS", pd.DataFrame(columns=["preferred_location_id", "x", "y"])
)
preferred_mode = globals().get("PREFERRED_MODE", "none")
preferred_match_report = pd.DataFrame()
if preferred_mode != "none" and len(preferred_locations):
    if len(preferred_locations) > len(target_instances):
        raise ValueError("Entered locations exceed the number of target instances.")
    eligible_tree = cKDTree(coords[eligible_indices])
    used_matches: set[int] = set()
    match_records: List[Dict[str, Any]] = []
    for _, preferred in preferred_locations.iterrows():
        k = min(len(eligible_indices), max(10, len(preferred_locations) * 5))
        while True:
            distances, local_positions = eligible_tree.query(
                np.array([preferred["x"], preferred["y"]], dtype=float), k=k
            )
            distances = np.atleast_1d(distances)
            local_positions = np.atleast_1d(local_positions).astype(int)
            available = [
                (float(distance), int(eligible_indices[position]))
                for distance, position in zip(distances, local_positions)
                if int(eligible_indices[position]) not in used_matches
            ]
            if available:
                geographic_distance, analysis_index = available[0]
                break
            if k >= len(eligible_indices):
                raise RuntimeError("Could not match entered locations to unique eligible survey observations.")
            k = min(len(eligible_indices), k * 2)
        used_matches.add(analysis_index)
        match_records.append(
            {
                "preferred_location_id": str(preferred["preferred_location_id"]),
                "input_x": float(preferred["x"]),
                "input_y": float(preferred["y"]),
                "matched_analysis_index": analysis_index,
                "matched_survey_x": float(coords[analysis_index, 0]),
                "matched_survey_y": float(coords[analysis_index, 1]),
                "distance_to_nearest_eligible_survey_observation": geographic_distance,
                "forced": preferred_mode == "force",
            }
        )
    preferred_match_report = pd.DataFrame(match_records)

    if preferred_mode == "force":
        locked_indices = preferred_match_report["matched_analysis_index"].to_numpy(dtype=int)
        lock_cost = np.linalg.norm(
            design_scores[locked_indices, None, :] - target_array[None, :, :], axis=2
        )
        lock_rows, lock_positions = linear_sum_assignment(lock_cost)
        for match_row, target_position in zip(lock_rows, lock_positions):
            locked_index = int(locked_indices[match_row])
            locked_distance = float(lock_cost[match_row, target_position])
            candidate_pools[target_position] = np.array([locked_index], dtype=int)
            candidate_pool_distances[target_position] = np.array([locked_distance], dtype=float)
            target_tolerances[target_position] = max(
                float(target_tolerances[target_position]), locked_distance
            )
            preferred_match_report.loc[match_row, "assigned_target_position"] = int(target_position)
            preferred_match_report.loc[match_row, "assigned_target_instance_id"] = str(
                target_instances.iloc[target_position]["target_instance_id"]
            )
            candidate_associations = candidate_associations[
                candidate_associations["target_position"] != target_position
            ]
            record = {
                "target_position": int(target_position),
                "target_instance_id": target_instances.iloc[target_position]["target_instance_id"],
                "base_target_id": target_instances.iloc[target_position]["base_target_id"],
                "target_type": target_instances.iloc[target_position]["target_type"],
                "candidate_analysis_index": locked_index,
                "candidate_rank": 0,
                "pca_target_distance": locked_distance,
                "final_tolerance_used": target_tolerances[target_position],
                "tolerance_expansions": -1,
                "preferred_location_id": preferred_match_report.iloc[match_row]["preferred_location_id"],
            }
            record.update(
                {column: float(target_instances.iloc[target_position][column]) for column in target_cols}
            )
            candidate_associations = pd.concat(
                [candidate_associations, pd.DataFrame([record])], ignore_index=True
            )
        minimum_assignment = assignment_from_costs(candidate_pools, candidate_pool_distances)
        if minimum_assignment is None:
            raise RuntimeError(
                "Forced existing locations made a unique assignment impossible. Increase candidate count/tolerance or use evaluate-only mode."
            )
        warnings.warn(
            "Forced existing locations are locked into the design. Inspect their PCA target mismatch and regression-design diagnostics."
        )
    display(preferred_match_report)

flat_candidates = np.concatenate(candidate_pools)
overlap_count = int(len(flat_candidates) - len(np.unique(flat_candidates)))
print(
    f"Candidate associations={len(flat_candidates):,}; unique candidate observations={len(np.unique(flat_candidates)):,}; "
    f"cross-pool duplicate associations={overlap_count:,}."
)
print(f"Final per-target PCA tolerance range: {min(target_tolerances):.4f} to {max(target_tolerances):.4f}")

final_selected, optimizer_summary = optimize_multiple_starts(
    minimum_assignment,
    candidate_pools,
    candidate_pool_distances,
    target_array,
    design_scores,
    coords,
    cfg,
)
display(optimizer_summary)

# Reproducibility and assignment acceptance tests.
repeat_selected, repeat_summary = optimize_multiple_starts(
    minimum_assignment,
    candidate_pools,
    candidate_pool_distances,
    target_array,
    design_scores,
    coords,
    cfg,
)
np.testing.assert_array_equal(final_selected, repeat_selected)
assert len(np.unique(final_selected)) == len(final_selected), "Selected survey IDs/rows are not unique."
assert len(final_selected) == len(target_instances), "Every target instance must receive exactly one site."
assert all(final_selected[i] in set(candidate_pools[i].tolist()) for i in range(len(final_selected)))
print("Optimizer reproducibility and unique one-to-one target assignment tests passed.")


### 4.5 Final design and regression diagnostics


In [ ]:
selected_coords = coords[final_selected].astype(np.float64)
selected_scores = design_scores[final_selected].astype(np.float64)
if len(preferred_match_report):
    preferred_match_report["selected_in_final_design"] = preferred_match_report["matched_analysis_index"].isin(final_selected)
nearest_selected = selected_nearest_neighbor_distances(selected_coords)
final_geo_msd = exact_geo_msd(selected_coords)
mean_target_distance, max_target_distance = matching_metrics(
    final_selected, target_array, design_scores
)
pc_balance = float(np.linalg.norm(np.mean(selected_scores, axis=0)))
regression_diagnostics, leverage = regression_design_diagnostics(
    selected_scores, design_scores[eligible_indices], cfg.DIAGNOSTIC_CHUNK_SIZE
)
final_diagnostics = {
    "geoMSD": final_geo_msd,
    "absolute_minimum_separation": float(np.min(nearest_selected)),
    "arithmetic_mean_nearest_neighbor_separation": float(np.mean(nearest_selected)),
    "median_nearest_neighbor_separation": float(np.median(nearest_selected)),
    "mean_pca_target_distance": mean_target_distance,
    "maximum_pca_target_distance": max_target_distance,
    "pc_balance_norm": pc_balance,
    "masked_pca_space_outliers": int(pca_outlier.sum()),
    "rows_within_design_radius": int(within_design_radius.sum()),
    "percent_within_design_radius": 100.0 * actual_design_coverage,
    **regression_diagnostics,
}
display(pd.DataFrame([final_diagnostics]))

# Figure 6: candidates and final sites in geographic space.
fig, ax = plt.subplots(figsize=(8, 6))
unique_candidate_indices = np.unique(flat_candidates)
ax.scatter(coords[unique_candidate_indices, 0], coords[unique_candidate_indices, 1], s=10, alpha=0.35, color=cfg.CANDIDATE_COLOR, label="Candidates")
ax.scatter(selected_coords[:, 0], selected_coords[:, 1], marker="x", s=70, color=cfg.SELECTED_COLOR, label="Final selected")
if len(preferred_match_report):
    ax.scatter(
        preferred_match_report["input_x"],
        preferred_match_report["input_y"],
        marker="*",
        s=130,
        color=cfg.TARGET_COLOR,
        edgecolor="black",
        linewidth=0.5,
        label="Entered locations",
    )
ax.set_xlabel(f"{cfg.X_COLUMN} (projected units)")
ax.set_ylabel(f"{cfg.Y_COLUMN} (projected units)")
ax.set_title("Candidate and final selected sites")
ax.legend()
fig.tight_layout()
plt.show()

# Figure 7: selected sites over every complete survey coordinate.
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=2, alpha=0.10, color=cfg.FOOTPRINT_COLOR, rasterized=True, label="Complete survey footprint")
ax.scatter(selected_coords[:, 0], selected_coords[:, 1], marker="x", s=70, color=cfg.SELECTED_COLOR, label="Selected calibration sites")
for i, row in target_instances.iterrows():
    ax.annotate(row["target_instance_id"], selected_coords[i], xytext=(3, 3), textcoords="offset points", fontsize=7)
if len(preferred_match_report):
    ax.scatter(
        preferred_match_report["input_x"],
        preferred_match_report["input_y"],
        marker="*",
        s=130,
        color=cfg.TARGET_COLOR,
        edgecolor="black",
        linewidth=0.5,
        label="Entered locations",
    )
ax.set_xlabel(f"{cfg.X_COLUMN} (projected units)")
ax.set_ylabel(f"{cfg.Y_COLUMN} (projected units)")
ax.set_title("Final selected locations over survey footprint")
ax.legend()
fig.tight_layout()
plt.show()

# Figure 8: selected observations linked to theoretical targets in PC space.
fig, ax = plt.subplots(figsize=(7.5, 6.2 if p > 1 else 4.8))
if p == 1:
    ax.scatter(
        target_array[:, 0],
        np.zeros(len(target_array)),
        marker="x",
        s=70,
        color=cfg.TARGET_COLOR,
        label="Theoretical targets",
    )
    ax.scatter(
        selected_scores[:, 0],
        np.zeros(len(selected_scores)),
        s=35,
        color=cfg.SELECTED_COLOR,
        label="Selected observations",
    )
    for target, observed in zip(target_array[:, 0], selected_scores[:, 0]):
        ax.plot([target, observed], [0, 0], linewidth=0.8, alpha=0.6)
    ax.set_xlabel("Standardized PC1")
    ax.set_yticks([])
    ax.set_title("Final one-dimensional response-surface target matches")
else:
    ax.scatter(
        target_array[:, 0],
        target_array[:, 1],
        marker="x",
        s=70,
        color=cfg.TARGET_COLOR,
        label="Theoretical targets",
    )
    ax.scatter(
        selected_scores[:, 0],
        selected_scores[:, 1],
        s=35,
        color=cfg.SELECTED_COLOR,
        label="Selected observations",
    )
    for target, observed in zip(target_array[:, :2], selected_scores[:, :2]):
        ax.plot([target[0], observed[0]], [target[1], observed[1]], linewidth=0.7, alpha=0.6)
    ax.set_xlabel("Standardized PC1")
    ax.set_ylabel("Standardized PC2")
    ax.set_title("Final response-surface target matches" + projection_note)
    ax.set_aspect("equal", adjustable="box")
ax.legend()
fig.tight_layout()
plt.show()

# Figure 9: nearest-selected-neighbor separations.
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.hist(nearest_selected, bins=min(12, max(5, len(nearest_selected) // 2)))
ax.axvline(final_geo_msd, linestyle="--", linewidth=1.5, label=f"geoMSD={final_geo_msd:.2f}")
ax.set_xlabel("Nearest selected neighbor distance (projected units)")
ax.set_ylabel("Selected sites")
ax.set_title("Distribution of selected-site nearest-neighbor distances")
ax.legend()
fig.tight_layout()
plt.show()

# Figure 10: optimizer improvement across starts.
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(optimizer_summary["start_number"], optimizer_summary["initial_geoMSD"], marker=".", label="Initial")
ax.plot(optimizer_summary["start_number"], optimizer_summary["final_geoMSD"], marker=".", label="Converged")
ax.set_xlabel("Optimizer start")
ax.set_ylabel("geoMSD (projected units)")
ax.set_title("Optimizer improvement across starts")
ax.legend()
fig.tight_layout()
plt.show()


### 4.6 Compare selected sites with the whole eligible survey

These boxplots compare standardized PC distributions and original sensor-feature distributions. They are diagnostics, not replacement objectives.


In [ ]:
# @title Boxplots: final selected sites versus whole eligible survey
fig, axes = plt.subplots(1, p, figsize=(4.8 * p, 5), squeeze=False)
for j in range(p):
    axes[0, j].boxplot(
        [design_scores[eligible_indices, j], selected_scores[:, j]], patch_artist=True
    )
    axes[0, j].set_xticks([1, 2], ["Whole eligible", "Selected"])
    axes[0, j].set_title(f"Standardized PC{j + 1}")
    axes[0, j].grid(axis="y", linestyle="--", alpha=0.35)
fig.suptitle("PCA distributions: whole eligible survey vs selected sites")
fig.tight_layout()
plt.show()

n_features = len(cfg.FEATURE_COLUMNS)
ncols = min(3, n_features)
nrows = math.ceil(n_features / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows), squeeze=False)
for j, feature in enumerate(cfg.FEATURE_COLUMNS):
    whole_values = pd.to_numeric(
        source_df.loc[analysis_to_source[eligible_indices], feature], errors="coerce"
    ).to_numpy(dtype=float)
    selected_values = pd.to_numeric(
        source_df.loc[analysis_to_source[final_selected], feature], errors="coerce"
    ).to_numpy(dtype=float)
    axes.flat[j].boxplot([whole_values, selected_values], patch_artist=True)
    axes.flat[j].set_xticks([1, 2], ["Whole eligible", "Selected"])
    axes.flat[j].set_title(feature)
    axes.flat[j].grid(axis="y", linestyle="--", alpha=0.35)
for ax in axes.flat[n_features:]:
    ax.set_visible(False)
fig.suptitle("Original sensor features: whole eligible survey vs selected sites")
fig.tight_layout()
plt.show()


### 4.7 Export selected sites, candidate associations, metadata, and run summary


In [ ]:
selected_source_rows = analysis_to_source[final_selected]
selected_export = source_df.loc[
    selected_source_rows,
    [cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, *cfg.FEATURE_COLUMNS],
].reset_index(drop=True)
for j in range(p):
    selected_export[f"PC{j + 1}"] = selected_scores[:, j]
selected_export["pca_space_outlier_flag"] = pca_outlier[final_selected]
selected_export["target_instance_id"] = target_instances["target_instance_id"].to_numpy()
selected_export["base_target_id"] = target_instances["base_target_id"].to_numpy()
selected_export["target_type"] = target_instances["target_type"].to_numpy()
selected_export["target_replicate_number"] = target_instances["target_replicate_number"].to_numpy()
for j in range(p):
    selected_export[f"theoretical_target_PC{j + 1}"] = target_array[:, j]
    selected_export[f"selected_observation_PC{j + 1}"] = selected_scores[:, j]
selected_export["pca_target_distance"] = np.linalg.norm(selected_scores - target_array, axis=1)
selected_export["nearest_selected_geographic_neighbor_distance"] = nearest_selected
selected_export["sample_role"] = "calibration"
selected_export["matched_preferred_location_id"] = pd.NA
selected_export["preferred_site_forced"] = False
if len(preferred_match_report):
    preferred_lookup = preferred_match_report.set_index("matched_analysis_index")
    for row_position, analysis_index in enumerate(final_selected):
        if int(analysis_index) in preferred_lookup.index:
            matched = preferred_lookup.loc[int(analysis_index)]
            selected_export.loc[row_position, "matched_preferred_location_id"] = matched["preferred_location_id"]
            selected_export.loc[row_position, "preferred_site_forced"] = bool(matched["forced"])

candidate_export = candidate_associations.copy()
candidate_analysis_indices = candidate_export.pop("candidate_analysis_index").to_numpy(dtype=int)
candidate_source_rows = analysis_to_source[candidate_analysis_indices]
candidate_identity = source_df.loc[
    candidate_source_rows,
    [cfg.ID_COLUMN, cfg.X_COLUMN, cfg.Y_COLUMN, *cfg.FEATURE_COLUMNS],
].reset_index(drop=True)
candidate_export = pd.concat([candidate_identity, candidate_export.reset_index(drop=True)], axis=1)
for j in range(p):
    candidate_export[f"candidate_PC{j + 1}"] = design_scores[candidate_analysis_indices, j]
candidate_export["pca_space_outlier_flag"] = pca_outlier[candidate_analysis_indices]

selected_export.to_csv("ESAP_RSSD_selected_sites.csv", index=False)
candidate_export.to_csv("ESAP_RSSD_candidate_sites.csv", index=False)

runtime_seconds = time.perf_counter() - run_started
metadata = {
    "notebook_version": NOTEBOOK_VERSION,
    "input_file": input_label,
    "configuration": asdict(cfg),
    "data_validation": validation_report,
    "rows_analyzed": int(len(design_scores)),
    "candidate_eligible_rows": int(len(eligible_indices)),
    "transformations": transformation_details,
    "feature_skewness": skewness_summary.reset_index(names="feature").to_dict(orient="records"),
    "pca_mode": pca_mode,
    "pca_explained_variance": np.asarray(pca_estimator.explained_variance_).tolist(),
    "pca_explained_variance_ratio": explained.tolist(),
    "pca_validation": {
        "means": pca_validation["mean"].tolist(),
        "sample_standard_deviations": pca_validation["sample_sd"].tolist(),
        "correlation_matrix": pca_correlation.tolist(),
    },
    "memory_report": memory_report,
    "outlier_screen": {
        "method": "sum_of_squared_standardized_design_PC_scores",
        "degrees_of_freedom": p,
        "coverage": cfg.OUTLIER_COVERAGE,
        "chi_square_d2_threshold": outlier_threshold_d2,
        "masked_count": int(pca_outlier.sum()),
    },
    "response_surface_design": {
        **design_report,
        "sample_budget_mode": cfg.SAMPLE_BUDGET_MODE,
        "replication_counts": target_replication_counts.to_dict(),
    },
    "candidate_discovery": {
        "candidate_associations": int(len(candidate_export)),
        "unique_candidate_observations": int(candidate_export[cfg.ID_COLUMN].nunique()),
        "cross_pool_duplicate_associations": overlap_count,
        "assignment_attempts": assignment_attempts,
        "target_tolerances": target_tolerances,
    },
    "optimizer": {
        "criterion": "lexicographic: maximize exact geoMSD; tie-break by mean then maximum PCA target distance",
        "starts": cfg.N_OPTIMIZER_STARTS,
        "random_seed": cfg.RANDOM_SEED,
        "convergence_summary": optimizer_summary.to_dict(orient="records"),
    },
    "approximate_candidate_prefilter": bool(approximate_prefilter_used),
    "approximate_prefilter_report": prefilter_report,
    "preferred_location_mode": preferred_mode,
    "preferred_location_matches": preferred_match_report.to_dict(orient="records") if len(preferred_match_report) else [],
    "coordinate_epsg": globals().get("COORDINATE_EPSG"),
    "final_diagnostics": final_diagnostics,
    "selected_nearest_neighbor_distances": nearest_selected.tolist(),
    "package_versions": package_versions(),
    "runtime_seconds": runtime_seconds,
}
Path("ESAP_RSSD_run_metadata.json").write_text(
    json.dumps(json_ready(metadata), indent=2, allow_nan=False), encoding="utf-8"
)

ave_pvar = regression_diagnostics.get("average_relative_prediction_variance")
run_summary = f"""# ESAP RSSD run summary

- **Input and records:** `{input_label}`; {len(source_df):,} source rows and {len(design_scores):,} complete finite rows analyzed.
- **Sensor variables:** {', '.join(cfg.FEATURE_COLUMNS)}.
- **Transformations:** {json.dumps({c: transformation_details[c]['method'] for c in cfg.FEATURE_COLUMNS})}.
- **PCA:** {p} design components; mode `{pca_mode}`; retained explained variance ratios {np.round(explained[:p], 6).tolist()}.
- **PCA standardization validation:** maximum absolute mean {mean_abs:.3g}, maximum SD error {sd_error:.3g}, maximum absolute off-diagonal correlation {max_offdiag:.3g}.
- **PCA-space outliers:** {int(pca_outlier.sum()):,} masked using chi-square D2 coverage {cfg.OUTLIER_COVERAGE} with {p} degrees of freedom.
- **Response-surface design:** full {p}-D central composite design; empirical radius R={design_radius:.6g}; actual coverage {actual_design_coverage:.4%}; {len(base_targets)} base targets and {len(target_instances)} target instances.
- **Sample budget:** `{cfg.SAMPLE_BUDGET_MODE}`; target replication counts {target_replication_counts.to_dict()}.
- **Candidate procedure:** cKDTree search in standardized PC space; closest candidate first, then greedy geographic separation within deterministically expanded target tolerances.
- **Optimization:** {cfg.N_OPTIMIZER_STARTS} reproducible unique starts; coordinate exchange maximized exact geoMSD, with target mismatch used only for effective ties.
- **Existing locations:** mode `{preferred_mode}`; {len(preferred_match_report)} entered location(s) matched to eligible survey observations.
- **Approximate candidate prefilter:** {approximate_prefilter_used}.
- **Spatial result:** geoMSD={final_geo_msd:.6g}; absolute minimum separation={float(np.min(nearest_selected)):.6g} projected coordinate units.
- **PCA target matching:** mean distance={mean_target_distance:.6g}; maximum distance={max_target_distance:.6g} standardized PC units.
- **PC balance:** Euclidean norm of mean selected design PC scores={pc_balance:.6g}.
- **Regression design:** rank {regression_diagnostics['matrix_rank']}/{regression_diagnostics['matrix_columns']}; condition number={regression_diagnostics['condition_number']:.6g}; maximum leverage={regression_diagnostics.get('maximum_leverage')}; avePVar={ave_pvar}.
- **Inference warning:** RSSD is a model-based directed sampling design. These selected sites alone are not a probability sample for unbiased finite-population estimation.
- **Future residual check:** Residual spatial independence must be assessed after response data are collected and a calibration model is fitted.
"""
Path("ESAP_RSSD_run_summary.md").write_text(run_summary, encoding="utf-8")

print("Wrote ESAP_RSSD_selected_sites.csv")
print("Wrote ESAP_RSSD_candidate_sites.csv")
print("Wrote ESAP_RSSD_run_metadata.json")
print("Wrote ESAP_RSSD_run_summary.md")
display(selected_export)


## Why Moran's I Is Not Used Here

Lesch (2005) uses the Moran residual statistic to assess spatial autocorrelation after a calibration response has been measured and a regression model has been fitted. At the RSSD site-selection stage, no target response or fitted calibration residuals exist. Standardized PCA scores are coded sensor covariates, not regression residuals. Consequently, Moran's I is neither calculated nor optimized here. It belongs in the later calibration and model-validation workflow, alongside other residual diagnostics.


## 3. Optional disabled 300,000-row stress test

This test exercises standardized PCA and exact tree-based candidate-neighborhood queries on correlated synthetic features. It reports elapsed time and major-array memory. It is disabled by default and does not alter the completed operational run.


In [ ]:
def run_large_data_stress_test(config: RSSDConfig) -> Optional[Dict[str, Any]]:
    """Run an optional approximately 300k-row scaling test without any N-by-N structure."""
    if not config.RUN_LARGE_STRESS_TEST:
        print("Large-data stress test disabled (RUN_LARGE_STRESS_TEST=False).")
        return None
    started = time.perf_counter()
    rng = np.random.default_rng(config.RANDOM_SEED + 1)
    n = config.STRESS_TEST_ROWS
    latent = rng.normal(size=(n, 3)).astype(np.float32)
    mixing = np.array(
        [[1.0, 0.8, -0.2, 0.4, 0.6, -0.3], [0.4, 1.1, 0.7, -0.5, 0.2, 0.8], [-0.3, 0.1, 1.0, 0.9, -0.4, 0.5]],
        dtype=np.float32,
    )
    features = latent @ mixing + rng.normal(0, 0.25, size=(n, 6)).astype(np.float32)
    standardized = StandardScaler().fit_transform(features).astype(np.float32, copy=False)
    estimator = PCA(n_components=3, svd_solver="randomized", random_state=config.RANDOM_SEED).fit(standardized)
    scores = estimator.transform(standardized) / np.sqrt(estimator.explained_variance_)
    tree = cKDTree(scores.astype(np.float32, copy=False))
    query_counts = [len(tree.query_ball_point(t, r=config.PC_CANDIDATE_TOLERANCE)) for t in np.eye(3)]
    arrays = {
        "latent_mib": latent.nbytes / 1024**2,
        "features_mib": features.nbytes / 1024**2,
        "standardized_mib": standardized.nbytes / 1024**2,
        "scores_mib": scores.nbytes / 1024**2,
    }
    report = {
        "rows": n,
        "features": 6,
        "elapsed_seconds": time.perf_counter() - started,
        "major_array_memory_mib": arrays,
        "target_neighborhood_counts": query_counts,
        "full_n_by_n_matrix_created": False,
    }
    display(pd.DataFrame([report]))
    del latent, features, standardized, scores, tree
    gc.collect()
    return report


stress_test_report = run_large_data_stress_test(cfg)
